# Author Contribution Assignment to CRediT: Rules + Few-shot Embeddings

This notebook runs a staged classification pipeline:

1. Load the data.
2. Assign roles using explicit labels, high-precision text rules, action–object rules, nominal phrases, and contextual corrections.
3. Create author-level and author–task-level outputs.
4. Select only tasks where `credit_send_to_few_shot=True`.
5. Load manually labeled few-shot examples.
6. Create an embedding for every example and a centroid for every CRediT category.
7. Measure cosine similarity between every unresolved task and every category.
8. Assign a semantic role only when the score passes a validated threshold.
9. Save the top score, distance, second-best category, score margin, and nearest example for auditing.

**Important:** the embedding stage never overwrites tasks already assigned confidently by the rule engine. It runs only on tasks explicitly marked for few-shot classification.


## 1. Install Libraries and Connect Google Drive

This cell installs the required packages. In Google Colab, uncomment the Drive lines to access files stored in Google Drive.

`sentence-transformers` creates the embeddings, and `scikit-learn` is available for validation metrics and threshold selection.


In [ ]:
# 1. Install Libraries and Connect Google Drive

# Run once in Google Colab.
!pip -q install spacy sentence-transformers scikit-learn openpyxl

# Optional: mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## 2. Load spaCy and Basic Settings

This cell imports the required packages and loads the English spaCy model.

If the full model is unavailable, the text-rule layers can still run with a blank English tokenizer, but dependency-based action–object rules will be limited.


In [ ]:
# 2. Load spaCy and Basic Settings

import json
import os
import re
import subprocess
import sys
import unicodedata
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from google.colab import files

import ast
import json
import shutil
import zipfile

import numpy as np

try:
    import spacy
    from spacy.tokens import Doc, Token
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'spacy'])
    import spacy
    from spacy.tokens import Doc, Token
SPACY_MODEL = 'en_core_web_sm'

def load_spacy_model(model_name: str=SPACY_MODEL):
    """Implement load spacy model. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    if os.environ.get('CREDIT_OFFLINE', '0') == '1':
        return spacy.blank('en')
    try:
        return spacy.load(model_name)
    except OSError:
        try:
            subprocess.check_call([sys.executable, '-m', 'spacy', 'download', model_name])
            return spacy.load(model_name)
        except Exception:
            return spacy.blank('en')
nlp = load_spacy_model()


## 3. CRediT Categories, Documentation, and Text Normalization

This cell defines the 14 CRediT roles, a concise explanation of how each role is determined, and the normalization functions used throughout the notebook.

It also recognizes explicit labels such as `Methodology`, `Formal Analysis`, or a list containing several explicit CRediT roles. Explicit labels receive the highest confidence.


In [ ]:
# 3. CRediT Categories, Documentation, and Text Normalization

CREDIT_ROLES = ['Conceptualization', 'Methodology', 'Investigation', 'Formal Analysis', 'Data Curation', 'Writing – Original Draft', 'Writing – Review & Editing', 'Supervision', 'Validation', 'Project Administration', 'Resources', 'Funding Acquisition', 'Visualization', 'Software']
CREDIT_ROLE_RULE_SUMMARY = {'Conceptualization': {'criterion': 'The research idea, research questions or hypotheses, concepts, and theory development.', 'examples': 'conceived the study; developed the theory; concept and design'}, 'Methodology': {'criterion': 'Research design, methods, protocols, models, and analysis plans.', 'examples': 'study design; developed the methodology; analysis plan'}, 'Investigation': {'criterion': 'Execution of the research and evidence collection, including experiments, measurements, samples, sequencing, and literature review.', 'examples': 'performed experiments; acquisition of data; fabricated devices'}, 'Formal Analysis': {'criterion': 'Analysis, calculation, modeling, interpretation, or scientific discussion of results.', 'examples': 'analyzed the data; performed bioinformatics; discussed the results'}, 'Data Curation': {'criterion': 'Preparation, cleaning, organization, assembly, documentation, and maintenance of data.', 'examples': 'compiled the data; data preparation; collection and assembly of data'}, 'Writing – Original Draft': {'criterion': 'Writing or preparing the original draft and manuscript sections.', 'examples': 'wrote; manuscript draft; drafted the paper'}, 'Writing – Review & Editing': {'criterion': 'Reviewing, revising, or editing the manuscript; feedback and comments require manuscript context.', 'examples': 'revised; edited; commented on the manuscript'}, 'Supervision': {'criterion': 'Active supervision, mentoring, or training people; passive supervision phrases and model training do not qualify.', 'examples': 'supervised the study; mentored students; trained researchers'}, 'Validation': {'criterion': 'Validation, replication, quality control, and checks of accuracy or reliability.', 'examples': 'validation; verified the results; quality checking'}, 'Project Administration': {'criterion': 'Research management and coordination, logistics, and participant recruitment.', 'examples': 'project management; coordinated the study; patient recruitment'}, 'Resources': {'criterion': 'Provision of materials, samples, equipment, facilities, tools, or existing datasets.', 'examples': 'provided samples; supplied reagents; material support'}, 'Funding Acquisition': {'criterion': 'Acquiring, securing, or preparing funding and grant applications.', 'examples': 'obtained funding; secured a grant; funding acquisition'}, 'Visualization': {'criterion': 'Creation or preparation of figures, graphs, maps, tables, and other visual material.', 'examples': 'prepared figures; visualization; figure preparation'}, 'Software': {'criterion': 'Development, implementation, programming, maintenance, or documentation of software and code.', 'examples': 'developed software; wrote code; implemented the algorithm'}}

def get_credit_role_summary() -> pd.DataFrame:
    """Implement get credit role summary. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    return pd.DataFrame([{'credit_role': role, 'determined_when': CREDIT_ROLE_RULE_SUMMARY[role]['criterion'], 'examples': CREDIT_ROLE_RULE_SUMMARY[role]['examples']} for role in CREDIT_ROLES])

def safe_text(value) -> str:
    """Implement safe text. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except (TypeError, ValueError):
        pass
    return str(value)

def normalize_task_text(value) -> str:
    """Implement normalize task text. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    text = unicodedata.normalize('NFKC', safe_text(value))
    text = text.replace('–', '-').replace('—', '-').replace('−', '-').replace('&', ' and ')
    text = text.lower().strip()
    text = re.sub('\\s+', ' ', text)
    text = text.strip(' \t\r\n.,;:')
    return text

def normalize_label_text(value) -> str:
    text = normalize_task_text(value)
    text = re.sub('[()\\[\\]{}]', ' ', text)
    text = re.sub('\\s*-\\s*', ' - ', text)
    text = re.sub('\\s+', ' ', text)
    return text.strip()
EXPLICIT_ROLE_ALIASES_RAW = {'Conceptualization': {'conceptualization', 'conceptualisation'}, 'Methodology': {'methodology', 'method development', 'development of methodology'}, 'Investigation': {'investigation'}, 'Formal Analysis': {'formal analysis', 'statistical analysis', 'mathematical analysis', 'computational analysis'}, 'Data Curation': {'data curation', 'data cleaning', 'data management'}, 'Writing – Original Draft': {'writing original draft', 'writing - original draft', 'writing original draft preparation', 'original draft preparation', 'writing - original draft preparation'}, 'Writing – Review & Editing': {'writing review and editing', 'writing - review and editing', 'writing review editing', 'review and editing', 'writing - review and editing preparation'}, 'Supervision': {'supervision'}, 'Validation': {'validation', 'data validation'}, 'Project Administration': {'project administration', 'project management', 'project coordination'}, 'Resources': {'resources', 'resource provision'}, 'Funding Acquisition': {'funding acquisition', 'acquisition of funding', 'obtaining funding'}, 'Visualization': {'visualization', 'visualisation', 'data visualization', 'data visualisation'}, 'Software': {'software', 'software development', 'code development'}}
EXPLICIT_ROLE_LOOKUP: Dict[str, str] = {}
for role, aliases in EXPLICIT_ROLE_ALIASES_RAW.items():
    for alias in aliases:
        EXPLICIT_ROLE_LOOKUP[normalize_label_text(alias)] = role

def detect_explicit_roles(normalized_task: str) -> List[str]:
    """Implement detect explicit roles. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    normalized_label = normalize_label_text(normalized_task)
    direct = EXPLICIT_ROLE_LOOKUP.get(normalized_label)
    if direct is not None:
        return [direct]
    parts = [normalize_label_text(part) for part in re.split('\\s*(?:;|,|\\||/)\\s*', normalized_task) if normalize_label_text(part)]
    if len(parts) <= 1:
        return []
    roles: List[str] = []
    for part in parts:
        role = EXPLICIT_ROLE_LOOKUP.get(part)
        if role is None:
            return []
        if role not in roles:
            roles.append(role)
    return roles


## 4. Special Statements and Standalone Noise

Not every contribution string represents a CRediT role.

This cell identifies approval statements such as `All authors approved the manuscript` and standalone noise tokens that should not be passed to semantic classification.


In [ ]:
# 4. Special Statements and Standalone Noise

STANDALONE_NOISE = {'while', 'whereas', 'however', 'but', 'then', 'especially', 'including', 'done', 'the', 'all', 'as', 'in addition', 'also'}
APPROVAL_PATTERNS = [re.compile('^(?:all authors )?(?:have )?(?:read|reviewed) and approved (?:the )?(?:final |submitted |published )?(?:article|manuscript|paper|version)(?: for publication)?$'), re.compile('^(?:all authors )?approved (?:the )?(?:final |submitted |published )?(?:article|manuscript|paper|version)$'), re.compile('^(?:all authors )?agreed to (?:the )?(?:final |published )?(?:article|manuscript|paper|version)$')]

def detect_special_type(normalized_task: str) -> Optional[str]:
    """Implement detect special type. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    for pattern in APPROVAL_PATTERNS:
        if pattern.fullmatch(normalized_task):
            return 'approval_statement'
    return None


## 5. Dependency-Parsing Utilities

This cell identifies action verbs, their objects, and their local grammatical context.

Example: in `collected, cleaned, and analyzed the data`, the coordinated verbs may share the object `data`. In contrast, `provided samples and contributed to data analysis` must not transfer `samples` to the verb `contributed`.


In [ ]:
# 5. Dependency-Parsing Utilities

VERB_POS = {'VERB', 'AUX'}
DIRECT_TARGET_DEPS = {'dobj', 'obj', 'attr', 'oprd', 'dative', 'nsubjpass', 'nsubj:pass'}
PREPOSITION_DEPS = {'prep', 'agent'}
PREPOSITION_OBJECT_DEPS = {'pobj', 'obj'}
PARTICIPATION_ACTIONS = {'assist', 'help', 'support', 'contribute', 'participate', 'collaborate', 'involve'}

def lemma(token: Token) -> str:
    """Implement lemma. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    value = token.lemma_.lower().strip()
    if not value or value == '-pron-':
        value = token.lower_
    return value

def content_lemmas(tokens: Iterable[Token]) -> Set[str]:
    """Implement content lemmas. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    values = set()
    for token in tokens:
        if token.is_space or token.is_punct:
            continue
        token_lemma = lemma(token)
        if token_lemma:
            values.add(token_lemma)
    return values

def ordered_unique_tokens(tokens: Iterable[Token]) -> List[Token]:
    """Implement ordered unique tokens. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    by_index = {token.i: token for token in tokens}
    return [by_index[index] for index in sorted(by_index)]

def subtree_without_conjunct_verbs(token: Token) -> List[Token]:
    """Implement subtree without conjunct verbs. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    result: List[Token] = []

    def visit(current: Token):
        result.append(current)
        for child in current.children:
            if child.dep_ == 'conj' and child.pos_ in VERB_POS:
                continue
            if child.dep_ in {'cc', 'punct'}:
                continue
            visit(child)
    visit(token)
    return ordered_unique_tokens(result)

def get_conj_verb_group(token: Token) -> List[Token]:
    """Implement get conj verb group. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    root = token
    while root.dep_ == 'conj' and root.head.pos_ in VERB_POS:
        root = root.head
    group: Dict[int, Token] = {root.i: root}
    stack = [root]
    while stack:
        current = stack.pop()
        for child in current.children:
            if child.dep_ == 'conj' and child.pos_ in VERB_POS:
                if child.i not in group:
                    group[child.i] = child
                    stack.append(child)
    return [group[index] for index in sorted(group)]

def get_direct_target_tokens(verb: Token) -> List[Token]:
    """Implement get direct target tokens. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    targets: List[Token] = []
    for child in verb.children:
        if child.dep_ in DIRECT_TARGET_DEPS:
            targets.extend(list(child.subtree))
            continue
        if child.dep_ in PREPOSITION_DEPS:
            for prep_child in child.children:
                if prep_child.dep_ in PREPOSITION_OBJECT_DEPS:
                    targets.extend(list(prep_child.subtree))
    return ordered_unique_tokens(targets)

def get_effective_target_tokens(verb: Token) -> Tuple[List[Token], bool]:
    """Implement get effective target tokens. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    own_targets = get_direct_target_tokens(verb)
    if own_targets:
        return (own_targets, False)
    if any((child.dep_ in PREPOSITION_DEPS for child in verb.children)):
        return ([], False)
    shared_targets: List[Token] = []
    for sibling in get_conj_verb_group(verb):
        if sibling.i == verb.i:
            continue
        sibling_targets = get_direct_target_tokens(sibling)
        if sibling_targets:
            shared_targets.extend(sibling_targets)
    return (ordered_unique_tokens(shared_targets), bool(shared_targets))

def get_predicate_context(verb: Token) -> Dict:
    """Implement get predicate context. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    component_tokens = subtree_without_conjunct_verbs(verb)
    target_tokens, used_shared_target = get_effective_target_tokens(verb)
    all_tokens = ordered_unique_tokens(list(component_tokens) + list(target_tokens))
    return {'verb': verb, 'action': lemma(verb), 'component_tokens': component_tokens, 'target_tokens': target_tokens, 'component_lemmas': content_lemmas(component_tokens), 'target_lemmas': content_lemmas(target_tokens), 'all_lemmas': content_lemmas(all_tokens), 'component_text': ' '.join((token.text for token in component_tokens)), 'target_text': ' '.join((token.text for token in target_tokens)), 'evidence_text': ' '.join((token.text for token in all_tokens)), 'used_shared_target': used_shared_target}

def get_action_tokens(doc: Doc) -> List[Token]:
    """Implement get action tokens. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    action_tokens: List[Token] = []
    known_actions = set()
    for rule in ACTION_TARGET_RULES:
        known_actions.update(rule.actions)
    for token in doc:
        token_lemma = lemma(token)
        if token.pos_ == 'VERB':
            action_tokens.append(token)
            continue
        if token.pos_ == 'AUX' and token_lemma in known_actions:
            action_tokens.append(token)
    return action_tokens


## 6. Action + Object Rules

This cell defines deterministic action–object rules.

Examples:

- `analyze + data/results` → **Formal Analysis**
- `collect + data/samples` → **Investigation**
- `provide + samples/equipment` → **Resources**
- `write + manuscript/paper` → **Writing – Original Draft**
- `provide + funding/fundings/grant` → **Funding Acquisition**

A rule may also require supporting context or forbid misleading context.


In [ ]:
# 6. Action + Object Rules

@dataclass(frozen=True)
class ActionTargetRule:
    role: str
    name: str
    actions: frozenset
    targets: frozenset
    required_context: frozenset = frozenset()
    forbidden_context: frozenset = frozenset()
    allow_without_target: bool = False

def rule(role: str, name: str, actions: Sequence[str], targets: Sequence[str], required_context: Sequence[str]=(), forbidden_context: Sequence[str]=(), allow_without_target: bool=False) -> ActionTargetRule:
    return ActionTargetRule(role=role, name=name, actions=frozenset(actions), targets=frozenset(targets), required_context=frozenset(required_context), forbidden_context=frozenset(forbidden_context), allow_without_target=allow_without_target)
ACTION_TARGET_RULES: List[ActionTargetRule] = [rule('Conceptualization', 'conceive_research', ['conceptualize', 'conceptualise', 'conceive'], ['study', 'research', 'project', 'idea', 'concept', 'work']), rule('Conceptualization', 'formulate_research_goal', ['formulate', 'propose', 'identify', 'refine', 'define', 'develop'], ['question', 'hypothesis', 'aim', 'objective', 'goal', 'idea', 'concept']), rule('Methodology', 'develop_method_or_design', ['develop', 'design', 'create', 'establish', 'modify', 'optimize', 'optimise', 'define', 'determine'], ['method', 'methodology', 'protocol', 'approach', 'framework', 'model', 'design', 'strategy', 'questionnaire', 'survey', 'trial', 'algorithm'], forbidden_context=['software', 'code', 'program', 'script', 'pipeline', 'figure', 'graph', 'chart', 'plot']), rule('Investigation', 'perform_research_activity', ['perform', 'conduct', 'execute', 'run', 'carry'], ['experiment', 'assay', 'measurement', 'fieldwork', 'interview', 'survey', 'trial', 'assessment'], forbidden_context=['analysis', 'simulation', 'quality', 'validation']), rule('Investigation', 'collect_or_generate_evidence', ['collect', 'gather', 'generate', 'record', 'measure', 'observe'], ['data', 'evidence', 'observation', 'sample', 'specimen', 'measurement']), rule('Investigation', 'literature_search_or_screening', ['search', 'screen', 'review'], ['literature', 'study', 'record', 'evidence'], forbidden_context=['manuscript', 'paper', 'article', 'draft']), rule('Investigation', 'test_hypothesis', ['test'], ['hypothesis']), rule('Data Curation', 'curate_data', ['clean', 'preprocess', 'scrub', 'annotate', 'label', 'catalog', 'catalogue', 'archive', 'preserve', 'harmonize', 'harmonise', 'integrate', 'merge', 'aggregate', 'deduplicate', 'organize', 'organise', 'format', 'structure', 'document', 'maintain', 'update', 'manage'], ['data', 'dataset', 'metadata', 'record', 'file', 'documentation', 'database'], forbidden_context=['project', 'study', 'software', 'equipment', 'instrument']), rule('Formal Analysis', 'analyze_or_interpret_results', ['analyze', 'analyse', 'interpret', 'synthesize', 'synthesise', 'evaluate', 'assess', 'compare', 'estimate', 'calculate', 'compute', 'derive'], ['data', 'result', 'finding', 'variable', 'dataset', 'measurement', 'pattern', 'relationship', 'outcome'], forbidden_context=['accuracy', 'reliability', 'reproducibility']), rule('Formal Analysis', 'perform_formal_analysis', ['perform', 'conduct', 'run', 'carry'], ['analysis', 'test', 'simulation', 'regression', 'modeling', 'modelling'], required_context=['statistical', 'mathematical', 'computational', 'bioinformatic', 'bioinformatics', 'regression', 'numerical', 'quantitative', 'qualitative', 'simulation']), rule('Formal Analysis', 'apply_model_for_analysis', ['apply', 'use', 'run'], ['model', 'algorithm'], required_context=['data', 'analysis', 'prediction', 'classification', 'result', 'outcome']), rule('Formal Analysis', 'interpret_results', ['interpret'], ['result', 'finding']), rule('Funding Acquisition', 'obtain_funding', ['acquire', 'secure', 'obtain', 'raise', 'provide'], ['funding', 'fund', 'grant', 'support'], required_context=['financial', 'funding', 'grant', 'money']), rule('Funding Acquisition', 'prepare_grant', ['prepare', 'develop', 'write', 'submit', 'coordinate'], ['proposal', 'application', 'submission', 'budget'], required_context=['grant', 'funding']), rule('Funding Acquisition', 'fund_study', ['fund'], ['study', 'research', 'project', 'work']), rule('Resources', 'provide_resources', ['provide', 'supply', 'contribute', 'donate', 'procure', 'transport'], ['material', 'reagent', 'sample', 'specimen', 'patient', 'animal', 'equipment', 'instrument', 'facility', 'resource', 'tool', 'dataset'], forbidden_context=['guidance', 'advice', 'feedback', 'funding']), rule('Resources', 'maintain_equipment', ['maintain', 'calibrate'], ['equipment', 'instrument', 'tool']), rule('Software', 'develop_software', ['program', 'code', 'implement', 'debug', 'document', 'maintain', 'optimize', 'optimise', 'share', 'develop', 'test'], ['software', 'code', 'program', 'script', 'pipeline', 'library', 'package', 'platform', 'application', 'infrastructure', 'system'], forbidden_context=['study', 'research', 'methodology', 'figure']), rule('Software', 'implement_algorithm', ['implement', 'code', 'program'], ['algorithm', 'model', 'pipeline']), rule('Software', 'write_code', ['write'], ['code', 'software', 'program', 'script']), rule('Supervision', 'supervise_or_mentor', ['supervise', 'oversee', 'mentor', 'advise', 'guide', 'train', 'teach'], ['project', 'study', 'research', 'work', 'team', 'researcher', 'student', 'experiment', 'analysis', 'methodology']), rule('Supervision', 'provide_guidance', ['provide'], ['guidance', 'mentorship', 'advice']), rule('Project Administration', 'manage_or_coordinate_project', ['coordinate', 'manage', 'administer', 'organize', 'organise', 'schedule', 'monitor'], ['project', 'study', 'trial', 'activity', 'workflow', 'timeline', 'budget', 'logistics', 'correspondence', 'progress', 'compliance'], forbidden_context=['data', 'metadata', 'software', 'equipment']), rule('Project Administration', 'recruit_participants', ['recruit', 'enroll', 'enrol'], ['participant', 'patient', 'subject']), rule('Validation', 'validate_or_verify_outputs', ['validate', 'verify', 'replicate', 'reproduce', 'benchmark', 'fact-check', 'cross-check', 'confirm'], ['data', 'result', 'finding', 'method', 'model', 'protocol', 'instrument', 'experiment', 'output', 'analysis', 'accuracy', 'reliability', 'reproducibility', 'integrity']), rule('Validation', 'quality_control', ['perform', 'conduct', 'carry'], ['control', 'validation', 'check'], required_context=['quality', 'technical', 'reproducibility']), rule('Validation', 'test_accuracy_or_reliability', ['test', 'check', 'confirm'], ['accuracy', 'reliability', 'reproducibility', 'integrity']), rule('Visualization', 'create_visual_material', ['visualize', 'visualise', 'create', 'prepare', 'generate', 'produce', 'design', 'draw', 'plot', 'render'], ['figure', 'graph', 'chart', 'plot', 'map', 'diagram', 'visualization', 'visualisation', 'video', 'table'], forbidden_context=['review', 'revision']), rule('Writing – Original Draft', 'write_original_manuscript', ['write', 'draft', 'author', 'compose', 'prepare'], ['manuscript', 'paper', 'article', 'draft', 'text', 'section', 'legend'], forbidden_context=['grant', 'proposal', 'application', 'code', 'software', 'figure', 'graph', 'chart', 'plot', 'review', 'revision', 'editing']), rule('Writing – Review & Editing', 'review_or_edit_manuscript', ['review', 'revise', 'edit', 'copy-edit', 'proofread', 'critique', 'criticize', 'refine', 'comment'], ['manuscript', 'paper', 'article', 'draft', 'text', 'language', 'figure', 'table', 'material'], forbidden_context=['literature', 'study', 'record', 'data', 'accuracy']), rule('Writing – Review & Editing', 'respond_to_reviewers', ['respond'], ['reviewer', 'comment', 'review']), rule('Writing – Review & Editing', 'provide_manuscript_feedback', ['provide'], ['feedback', 'comment'], required_context=['manuscript', 'paper', 'article', 'draft', 'text'])]


## 7. Match an Action–Object Rule

This cell checks four conditions:

1. The action matches the rule.
2. A required object is present.
3. Any required contextual term is present.
4. No forbidden contextual term is present.

The check uses both spaCy lemmas and normalized surface text, including equivalent forms such as `data/datum` and `analyze/analyse`.


In [ ]:
# 7. Match an Action–Object Rule

def term_present(term: str, context: Dict) -> bool:
    """Implement term present. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    term = normalize_task_text(term)
    normalized_evidence = normalize_task_text(context['evidence_text'])
    if ' ' in term:
        return re.search(f'(?<![a-z]){re.escape(term)}(?![a-z])', normalized_evidence) is not None
    if term in context['all_lemmas']:
        return True
    if re.search(f'(?<![a-z]){re.escape(term)}(?![a-z])', normalized_evidence):
        return True
    equivalent_forms = {'data': {'datum'}, 'datum': {'data'}, 'analyze': {'analyse'}, 'analyse': {'analyze'}, 'modeling': {'modelling'}, 'modelling': {'modeling'}, 'visualization': {'visualisation'}, 'visualisation': {'visualization'}, 'organization': {'organisation'}, 'organisation': {'organization'}}
    return bool(equivalent_forms.get(term, set()) & set(context['all_lemmas']))

def any_term_present(terms: Iterable[str], context: Dict) -> bool:
    return any((term_present(term, context) for term in terms))

def match_action_target_rule(context: Dict, current_rule: ActionTargetRule) -> bool:
    """Implement match action target rule. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    if context['action'] not in current_rule.actions:
        return False
    if not current_rule.allow_without_target and (not any_term_present(current_rule.targets, context)):
        return False
    if current_rule.required_context:
        if not any_term_present(current_rule.required_context, context):
            return False
    if current_rule.forbidden_context:
        if any_term_present(current_rule.forbidden_context, context):
            return False
    return True


## 8. Nominal Phrases Without an Explicit Verb

Some contribution statements are short labels rather than complete clauses, such as `data cleaning`, `study design`, or `figure preparation`.

This cell maps such nominal phrases directly to CRediT roles.


In [ ]:
# 8. Nominal Phrases Without an Explicit Verb

NOMINAL_ROLE_PATTERNS_RAW = {'Conceptualization': ['\\bconceptuali[sz]ation\\b', '\\bresearch question formulation\\b', '\\bhypothesis development\\b', '\\bconceptual framework development\\b'], 'Methodology': ['\\bstudy design\\b', '\\bexperimental design\\b', '\\bmethod(?:ology)? development\\b', '\\bprotocol development\\b', '\\bsampling strategy\\b', '\\bsearch strategy\\b', '\\bmodel development\\b', '\\balgorithm development\\b'], 'Investigation': ['\\bdata collection\\b', '\\bevidence collection\\b', '\\bsample collection\\b', '\\bspecimen collection\\b', '\\bliterature search\\b', '\\bstudy screening\\b', '\\bfield ?work\\b', '\\bexperimental work\\b'], 'Data Curation': ['\\bdata curation\\b', '\\bdata cleaning\\b', '\\bdata preprocessing\\b', '\\bdata annotation\\b', '\\bdata management\\b', '\\bmetadata management\\b', '\\bdataset integration\\b', '\\bdata archiving\\b'], 'Formal Analysis': ['\\bformal analysis\\b', '\\bdata analysis\\b', '\\bstatistical analysis\\b', '\\bmathematical analysis\\b', '\\bcomputational analysis\\b', '\\bregression analysis\\b', '\\bbioinformatics analysis\\b', '\\bdata interpretation\\b', '\\bresults? interpretation\\b', '\\bnumerical simulations?\\b'], 'Funding Acquisition': ['\\bfunding acquisition\\b', '\\bgrant writing\\b', '\\bgrant proposal preparation\\b', '\\bfunding source identification\\b'], 'Resources': ['\\bresource provision\\b', '\\bmaterial provision\\b', '\\breagent provision\\b', '\\bsample provision\\b', '\\bequipment provision\\b', '\\bcomputing resources?\\b'], 'Software': ['\\bsoftware development\\b', '\\bcode development\\b', '\\bprogramming\\b', '\\bcoding\\b', '\\bpipeline development\\b', '\\bsoftware maintenance\\b'], 'Supervision': ['\\bsupervision\\b', '\\bmentorship\\b', '\\bresearch guidance\\b', '\\bscientific guidance\\b'], 'Project Administration': ['\\bproject administration\\b', '\\bproject management\\b', '\\bproject coordination\\b', '\\blogistics coordination\\b', '\\bparticipant recruitment\\b'], 'Validation': ['\\bvalidation\\b', '\\bverification\\b', '\\breplication\\b', '\\breproducibility assessment\\b', '\\bquality control\\b', '\\btechnical validation\\b'], 'Visualization': ['\\bvisuali[sz]ation\\b', '\\bfigure preparation\\b', '\\bgraph creation\\b', '\\bplot generation\\b', '\\btable preparation\\b'], 'Writing – Original Draft': ['\\boriginal draft\\b', '\\bmanuscript drafting\\b', '\\bmanuscript writing\\b', '\\bwriting of the manuscript\\b', '\\bfirst draft preparation\\b'], 'Writing – Review & Editing': ['\\bwriting review and editing\\b', '\\breview and editing\\b', '\\bmanuscript revision\\b', '\\bmanuscript editing\\b', '\\blanguage editing\\b', '\\bproofreading\\b']}
NOMINAL_ROLE_PATTERNS = {role: [re.compile(pattern) for pattern in patterns] for role, patterns in NOMINAL_ROLE_PATTERNS_RAW.items()}

def match_nominal_roles(normalized_task: str) -> Tuple[List[str], List[Dict]]:
    """Implement match nominal roles. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    roles: List[str] = []
    matches: List[Dict] = []
    for role, patterns in NOMINAL_ROLE_PATTERNS.items():
        for pattern in patterns:
            match = pattern.search(normalized_task)
            if match is None:
                continue
            if role not in roles:
                roles.append(role)
            matches.append({'role': role, 'rule': 'nominal_phrase', 'evidence': match.group(0)})
    return (roles, matches)


## 9. Contribution, Assistance, and Participation Verbs

Generic verbs such as `contributed`, `assisted`, and `participated` are not sufficient by themselves.

The assigned role is determined by the activity they modify. For example, `contributed to data analysis` may receive **Formal Analysis**, whereas `contributed to the work` remains too general for a CRediT assignment.


In [ ]:
# 9. Contribution, Assistance, and Participation Verbs

def classify_participation_predicate(context: Dict) -> Tuple[List[str], List[Dict]]:
    """Implement classify participation predicate. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    if context['action'] not in PARTICIPATION_ACTIONS:
        return ([], [])
    normalized_evidence = normalize_task_text(context['evidence_text'])
    roles, nominal_matches = match_nominal_roles(normalized_evidence)
    matches = []
    for item in nominal_matches:
        matches.append({**item, 'rule': f"participation_via_{item['rule']}", 'action': context['action']})
    return (roles, matches)


## 10. Candidate Hints for Few-shot Classification

These patterns do not assign a role.

They only generate possible candidate categories for unresolved tasks. For example, the word `figure` may suggest Visualization, but it does not force the assignment without stronger evidence.


In [ ]:
# 10. Candidate Hints for Few-shot Classification

CANDIDATE_HINTS_RAW = {'Conceptualization': ['\\bconcept', '\\bidea\\b', '\\bhypothes', '\\bresearch question', '\\baims?\\b', '\\bobjectives?\\b'], 'Methodology': ['\\bmethod', '\\bprotocol', '\\bstudy design', '\\bframework', '\\bmodel\\b', '\\balgorithm'], 'Investigation': ['\\bexperiment', '\\bfield ?work\\b', '\\binterview', '\\bcollect', '\\bgather', '\\bsample', '\\bliterature'], 'Formal Analysis': ['\\banaly', '\\binterpret', '\\bstatistic', '\\bcomput', '\\bregression', '\\bsimulation'], 'Data Curation': ['\\bclean', '\\bcurat', '\\bmetadata', '\\bdataset', '\\bannotat', '\\barchive', '\\bdata management'], 'Writing – Original Draft': ['\\bdraft', '\\bwrote\\b', '\\bwriting\\b', '\\bmanuscript'], 'Writing – Review & Editing': ['\\breview', '\\brevis', '\\bedit', '\\bproofread', '\\bcomment', '\\bfeedback'], 'Supervision': ['\\bsupervis', '\\bmentor', '\\bguid', '\\badvis', '\\boversee'], 'Validation': ['\\bvalid', '\\bverif', '\\breplic', '\\breproduc', '\\bbenchmark'], 'Project Administration': ['\\badmin', '\\bcoordin', '\\blogistic', '\\btimeline', '\\brecruit', '\\bproject management'], 'Resources': ['\\bmaterial', '\\breagent', '\\bequipment', '\\bresource', '\\bspecimen', '\\bpatient sample'], 'Funding Acquisition': ['\\bfund', '\\bgrant', '\\bfinancial support', '\\bbudget'], 'Visualization': ['\\bvisual', '\\bfigure', '\\bgraph', '\\bplot', '\\bchart'], 'Software': ['\\bsoftware', '\\bcode\\b', '\\bcoding\\b', '\\bprogram', '\\bscript', '\\bpipeline']}
CANDIDATE_HINTS = {role: [re.compile(pattern) for pattern in patterns] for role, patterns in CANDIDATE_HINTS_RAW.items()}

def generate_candidate_roles(normalized_task: str) -> List[str]:
    """Implement generate candidate roles. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    scores: Dict[str, int] = {}
    for role, patterns in CANDIDATE_HINTS.items():
        score = sum((1 for pattern in patterns if pattern.search(normalized_task)))
        if score > 0:
            scores[role] = score
    return [role for role, _ in sorted(scores.items(), key=lambda item: (-item[1], CREDIT_ROLES.index(item[0])))]


## 11. High-Precision Text Rules

This layer runs before dependency parsing and handles short, frequent contribution phrases directly.

The task is not split on commas, semicolons, or `and`. If several clear activities appear in one task, the same task may receive multiple CRediT roles.


In [ ]:
# 11. High-Precision Text Rules

TEXT_ROLE_PATTERNS_RAW = {'Conceptualization': [('conceptualization', '\\bconceptuali[sz](?:e|ed|ing|ation)?\\b'), ('conceive', '\\bconceiv(?:e|ed|ing)\\b'), ('conception', '\\bconception\\b'), ('research_goal', '\\b(?:formulat(?:e|ed|ing)|propos(?:e|ed|ing)|develop(?:ed|ing)?|defin(?:e|ed|ing))\\b[^.;|]{0,50}\\b(?:hypothes(?:is|es)|research questions?|aims?|objectives?|concepts?|ideas?)\\b'), ('plan_study', '\\b(?:plan(?:ned|ning)?|devis(?:e|ed|ing)|initiat(?:e|ed|ing)|originat(?:e|ed|ing))\\b[^.;|]{0,35}\\b(?:study|research|project|work|idea|concept)\\b'), ('study_concept', '\\b(?:study|research|project)\\s+(?:concept|conception|planning)\\b'), ('formulate_problem', '\\bformulat(?:e|ed|ing)\\b[^.;|]{0,30}\\b(?:problem|theory|hypothesis|question)\\b')], 'Methodology': [('methodology', '\\bmethodolog(?:y|ical)\\b'), ('method_design', '\\bmethods?\\s+(?:development|design)\\b'), ('nominal_design', '\\b(?:study|research|experimental|experiment|trial|survey|questionnaire|protocol|sampling|model|algorithm)\\s+design\\b'), ('design_activity', '\\bdesign(?:ed|ing)?\\b[^.;|]{0,40}\\b(?:study|research|experiment(?:s)?|trial|survey|questionnaire|protocol|method(?:ology)?|model|algorithm|approach|framework|procedure)\\b'), ('develop_method', '\\b(?:develop(?:ed|ing)?|establish(?:ed|ing)?|optim(?:ize|ized|izing|ise|ised|ising)|modif(?:y|ied|ying))\\b[^.;|]{0,40}\\b(?:method(?:ology)?|protocol|approach|framework|model|algorithm|procedure|questionnaire|survey)\\b'), ('plan_experiment', '\\bplan(?:ned|ning)?\\b[^.;|]{0,30}\\b(?:experiment(?:s)?|protocol|method|study design)\\b'), ('devise_method', '\\bdevis(?:e|ed|ing)\\b[^.;|]{0,30}\\b(?:method|approach|protocol|strategy|experiment(?:s)?)\\b'), ('model_setup', '\\b(?:model|simulation)\\s+(?:development|design|setup)\\b'), ('analysis_plan', '(?:\\b(?:data|statistical|formal|qualitative|quantitative|computational)?\\s*analysis\\s+(?:plan|planning|protocol|strategy|framework)\\b|\\b(?:plan|planning|protocol|strategy|framework)\\s+(?:for|of)\\s+(?:the\\s+)?(?:data\\s+|statistical\\s+|formal\\s+)?analysis\\b)')], 'Investigation': [('perform_research', '\\b(?:perform(?:ed|ing)?|conduct(?:ed|ing)?|execut(?:e|ed|ing)|carried out|carry out|ran|run)\\b[^.;|]{0,35}\\b(?:experiment(?:s)?|assay(?:s)?|measurement(?:s)?|fieldwork|field work|interview(?:s)?|survey(?:s)?|trial(?:s)?|research|experimental work)\\b'), ('collect_evidence', '\\b(?:collect(?:ed|ing)?|gather(?:ed|ing)?|acquir(?:e|ed|ing)|obtain(?:ed|ing)?|generat(?:e|ed|ing)|record(?:ed|ing)?|measur(?:e|ed|ing)|observ(?:e|ed|ing)|recruit(?:ed|ing)?|enrol(?:led|ling)?|enroll(?:ed|ing)?)\\b[^.;|]{0,35}\\b(?:data|samples?|specimens?|measurements?|observations?|participants?|patients?|subjects?|evidence)\\b'), ('nominal_collection', '\\b(?:data|sample|specimen|evidence)\\s+(?:collection|acquisition|generation)\\b'), ('sample_work', '\\b(?:sample|specimen)\\s+(?:preparation|fabrication|processing)\\b'), ('prepare_samples', '\\b(?:fabricat(?:e|ed|ing)|prepar(?:e|ed|ing))\\b[^.;|]{0,30}\\b(?:samples?|specimens?)\\b'), ('experiments_only', '^\\s*experiments?\\s*$'), ('did_experiments', '\\b(?:did|undertook)\\b[^.;|]{0,20}\\b(?:experiment(?:s)?|research|assay(?:s)?)\\b'), ('assist_experiments', '\\bassist(?:ed|ing)?\\b[^.;|]{0,25}\\b(?:experiment(?:s)?|assay(?:s)?|data collection|fieldwork)\\b'), ('experimental_studies', '\\bexperimental studies\\b'), ('characterization', '\\bperform(?:ed|ing)?\\b[^.;|]{0,30}\\b(?:characterization(?:s)?|characterisation(?:s)?|investigation(?:s)?|synthesis)\\b'), ('literature_review', '(?:\\b(?:systematic\\s+|scoping\\s+|narrative\\s+)?literature\\s+(?:review|search|screening)\\b|\\breview(?:ed|ing)?\\s+(?:the\\s+)?literature\\b)')], 'Formal Analysis': [('analysis_word', '\\b(?:analy[sz](?:e|ed|ing)|analyses|analysis(?!\\s+(?:tools?|software|code|pipeline|methods?|plan|planning|protocol|strategy|framework)))\\b'), ('interpret_results', '\\b(?:interpret(?:ed|ing|ation)?|evaluat(?:e|ed|ing|ion)|assess(?:ed|ing|ment)|compar(?:e|ed|ing|ison)|calculat(?:e|ed|ing|ion)|comput(?:e|ed|ing|ation)|estimat(?:e|ed|ing|ion))\\b[^.;|]{0,40}\\b(?:data|results?|findings?|outcomes?|measurements?|patterns?|relationships?|variables?)\\b'), ('nominal_interpretation', '\\b(?:data|results?|findings?)\\s+interpretation\\b'), ('discuss_results', '\\bdiscuss(?:ed|ing)?\\b[^.;|]{0,40}\\b(?:results?|findings?|implications?|conclusions?)\\b'), ('results_discussion', '\\bdiscussion\\s+of\\s+(?:the\\s+)?(?:results?|findings?)\\b'), ('formal_method', '\\b(?:statistical|mathematical|computational|bioinformatic(?:s)?|quantitative|qualitative|regression|numerical)\\s+(?:analysis|analyses|modeling|modelling)\\b'), ('analysis_only', '^\\s*(?:analysis|analyses|interpretation|calculations?|simulations?|modeling|modelling)\\s*$'), ('perform_calculations', '\\b(?:perform(?:ed|ing)?|conduct(?:ed|ing)?|carry(?:ing)? out|carried out)\\b[^.;|]{0,35}\\b(?:calculations?|computations?|simulations?|modeling|modelling|theoretical work|dft)\\b'), ('theoretical_calculations', '\\b(?:dft|theoretical|numerical|computational)\\s+calculations?\\b'), ('model_results', '\\b(?:model(?:ed|led|ing)?|simulat(?:e|ed|ing)|solv(?:e|ed|ing))\\b[^.;|]{0,35}\\b(?:problem|results?|data|system|model|equations?)\\b')], 'Data Curation': [('nominal_data_curation', '\\bdata\\s+(?:curation|cleaning|preprocessing|processing|management|annotation|labelling|labeling|archiving|organization|organisation|integration|harmonization|harmonisation|documentation|maintenance)\\b'), ('curate_data', '\\b(?:curat(?:e|ed|ing)|clean(?:ed|ing)?|preprocess(?:ed|ing)?|process(?:ed|ing)?|annotat(?:e|ed|ing)|label(?:led|ed|ling|ing)|catalog(?:ued|ed|uing|ing)|archiv(?:e|ed|ing)|harmoni[sz](?:e|ed|ing)|integrat(?:e|ed|ing)|merg(?:e|ed|ing)|deduplicat(?:e|ed|ing)|organi[sz](?:e|ed|ing)|format(?:ted|ting)?|manag(?:e|ed|ing))\\b[^.;|]{0,35}\\b(?:data|datasets?|metadata|records?|files?|database)\\b'), ('prepare_data', '\\bprepar(?:e|ed|ing)\\b[^.;|]{0,25}\\b(?:data|dataset(?:s)?|database|metadata)\\b'), ('passive_data_processing', '\\b(?:data|datasets?)\\s+(?:were|was)?\\s*(?:harmoni[sz]ed|processed|prepared|organized|organised|curated|cleaned)\\b')], 'Writing – Original Draft': [('explicit_original_draft', '\\bwriting\\s*[-–—]?\\s*(?:the\\s+)?original draft\\b'), ('original_draft', '\\boriginal draft(?: preparation)?\\b'), ('write_manuscript', '\\b(?:writ(?:e|ing)|wrote|draft(?:ed|ing)?|author(?:ed|ing)?|compos(?:e|ed|ing)|prepar(?:e|ed|ing))\\b[^.;|]{0,35}\\b(?:manuscript|paper|article|draft|text|section|report)\\b'), ('nominal_manuscript_writing', '\\bmanuscript\\s+(?:writing|drafting|preparation)\\b'), ('writing_only', '^\\s*writing\\s*$'), ('led_writing', '\\b(?:lead|led|leading)\\b[^.;|]{0,25}\\b(?:writing|drafting|manuscript preparation)\\b'), ('contributed_writing', '\\bcontribut(?:e|ed|ing)\\b[^.;|]{0,30}\\b(?:writing|paper writing|manuscript preparation|preparation of (?:the )?(?:manuscript|paper|article)|drafting)\\b'), ('participated_writing', '\\bparticipat(?:e|ed|ing)\\b[^.;|]{0,25}\\b(?:writing|drafting|manuscript preparation)\\b'), ('cowrote', '\\b(?:co[- ]?wrote|cowrote|co[- ]?authored)\\b[^.;|]{0,20}\\b(?:manuscript|paper|article)?\\b'), ('paper_preparation', '\\b(?:manuscript|paper|article)\\s+(?:preparation|writing|drafting)\\b')], 'Writing – Review & Editing': [('explicit_review_editing', '\\bwriting\\s*[-–—]?\\s*review\\s*(?:and|&)?\\s*editing\\b'), ('review_and_editing', '\\breview\\s*(?:and|&)\\s*editing\\b'), ('review_manuscript', '\\b(?:review(?:ed|ing)?|revis(?:e|ed|ing|ion)|edit(?:ed|ing)?|proofread(?:ing)?|comment(?:ed|ing)?|critic(?:ally|al)?\\s+revis(?:e|ed|ing|ion)|critique(?:d|ing)?|refin(?:e|ed|ing))\\b[^.;|]{0,45}\\b(?:manuscript|paper|article|draft|text|submission|version)\\b'), ('nominal_revision', '\\b(?:manuscript|paper|article|draft)\\s+(?:revision|editing|review|proofreading)\\b'), ('critical_revision', '\\bcritical revision\\b'), ('critical_feedback_on_manuscript', '(?:\\bprovided\\b[^.;|]{0,25}\\b(?:critical|editorial|intellectual)\\b[^.;|]{0,25}\\b(?:feedback|input|comments?)\\b[^.;|]{0,45}\\b(?:manuscript|paper|article|draft|text|submission|version)\\b|\\b(?:manuscript|paper|article|draft|text|submission|version)\\b[^.;|]{0,45}\\b(?:feedback|input|comments?|suggestions?)\\b)'), ('final_manuscript_contribution', '\\bcontribut(?:e|ed|ing)\\b[^.;|]{0,35}\\b(?:final|revised)\\b[^.;|]{0,20}\\b(?:manuscript|paper|article|version)\\b'), ('editing_only', '^\\s*(?:editing|revision|reviewing and editing|review and edit|critical review)\\s*$'), ('improve_manuscript', '\\b(?:improv(?:e|ed|ing)|finali[sz](?:e|ed|ing)|refin(?:e|ed|ing))\\b[^.;|]{0,30}\\b(?:manuscript|paper|article|draft)\\b'), ('contributed_revision', '\\bcontribut(?:e|ed|ing)\\b[^.;|]{0,35}\\b(?:revision(?:s)?|editing|review|proof(?:s|reading)?)\\b'), ('participated_editing', '\\bparticipat(?:e|ed|ing)\\b[^.;|]{0,25}\\b(?:editing|revision|review)\\b'), ('provided_manuscript_comments', '(?:\\bprovid(?:e|ed|ing)\\b[^.;|]{0,30}\\b(?:edits?|comments?|feedback|suggestions?|revisions?)\\b[^.;|]{0,45}\\b(?:manuscript|paper|article|draft|text|submission|version)\\b|\\b(?:manuscript|paper|article|draft|text|submission|version)\\b[^.;|]{0,45}\\b(?:edits?|comments?|feedback|suggestions?|revisions?)\\b)'), ('read_manuscript', '\\b(?:reviewed|checked)\\b[^.;|]{0,30}\\b(?:manuscript|paper|article|draft|text)\\b'), ('discussed_manuscript', '\\bdiscuss(?:ed|ing)\\b[^.;|]{0,35}\\b(?:manuscript|paper|article|draft)\\b'), ('edited_pronoun', '\\b(?:revised|edited|reviewed|commented|proofread|improved|finalized|finalised)\\s+(?:it|this|the work)\\b')], 'Supervision': [('supervision', '\\b(?:supervis(?:e|ed|ing|ion)|oversee|oversaw|overseen|mentor(?:ed|ing|ship)?|guid(?:e|ed|ing|ance)|advis(?:e|ed|ing))\\b'), ('train_people', '\\btrain(?:ed|ing)?\\b[^.;|]{0,35}\\b(?:students?|researchers?|staff|personnel|trainees?|technicians?|team members?|junior colleagues?)\\b'), ('lead_research', '\\b(?:lead|led|leading|direct(?:ed|ing)?)\\b[^.;|]{0,30}\\b(?:project|study|research|work|team|consortium)\\b')], 'Validation': [('validation', '\\b(?:validat(?:e|ed|ing|ion)|verif(?:y|ied|ying|ication)|replicat(?:e|ed|ing|ion)|reproduc(?:e|ed|ing|ibility)|benchmark(?:ed|ing)?|cross[- ]check(?:ed|ing)?|fact[- ]check(?:ed|ing)?|quality control|technical validation)\\b')], 'Project Administration': [('project_administration', '\\bproject administration\\b'), ('project_management', '\\bproject (?:management|coordination)\\b'), ('manage_project', '\\b(?:coordinat(?:e|ed|ing|ion)|administ(?:er|ered|ering|ration)|manag(?:e|ed|ing)|organi[sz](?:e|ed|ing)|schedul(?:e|ed|ing)|monitor(?:ed|ing))\\b[^.;|]{0,40}\\b(?:project|study|trial|workflow|timeline|logistics|correspondence|progress)\\b'), ('implementation_logistics', '\\bimplementation and logistics\\b')], 'Resources': [('resources', '\\bresources?\\b'), ('provide_resources', '\\b(?:provid(?:e|ed|ing)|suppl(?:y|ied|ying)|donat(?:e|ed|ing)|procur(?:e|ed|ing)|contribut(?:e|ed|ing))\\b[^.;|]{0,35}\\b(?:materials?|reagents?|samples?|specimens?|patients?|animals?|equipment|instruments?|facilities|resources?|tools?|datasets?)\\b'), ('resource_provision', '\\b(?:material|reagent|sample|specimen|equipment|instrument|resource)\\s+provision\\b'), ('provide_data', '\\b(?:provid(?:e|ed|ing)|suppl(?:y|ied|ying))\\b[^.;|]{0,20}\\b(?:data|clinical data|source data)\\b'), ('contributed_data', '\\bcontribut(?:e|ed|ing)\\s+(?:the\\s+)?(?:clinical\\s+|source\\s+)?data\\b'), ('data_contributed', '\\bdata (?:were|was)?\\s*contribut(?:e|ed|ing)\\b'), ('data_contribution', '\\bdata contribution\\b'), ('provision_study_material', '\\bprovision of\\b[^.;|]{0,25}\\b(?:materials?|patients?|samples?|specimens?|resources?)\\b')], 'Funding Acquisition': [('funding_acquisition', '\\bfunding acquisition\\b'), ('obtain_funding', '\\b(?:acquir(?:e|ed|ing)|secur(?:e|ed|ing)|obtain(?:ed|ing)|rais(?:e|ed|ing)|provid(?:e|ed|ing)|receiv(?:e|ed|ing))\\b[^.;|]{0,35}\\b(?:fundings?|funds?|grants?|financial support)\\b'), ('funding_activity', '\\b(?:fund(?:ed|ing)?|grant writing|grant proposal|funding proposal)\\b')], 'Visualization': [('visualization', '\\bvisuali[sz](?:e|ed|ing|ation)\\b'), ('create_figures', '\\b(?:prepar(?:e|ed|ing)|creat(?:e|ed|ing)|generat(?:e|ed|ing)|produc(?:e|ed|ing)|design(?:ed|ing)?|draw|drew|plot(?:ted|ting)?)\\b[^.;|]{0,30}\\b(?:fig(?:ure)?s?|graphs?|charts?|plots?|maps?|diagrams?|tables?|visuals?)\\b'), ('nominal_visualization', '\\b(?:figure|graph|chart|plot|map|diagram|table)\\s+(?:preparation|creation|generation|design)\\b'), ('made_figures', '\\bmade\\b[^.;|]{0,20}\\b(?:fig(?:ure)?s?|graphs?|plots?|charts?)\\b')], 'Software': [('software', '\\bsoftware\\b'), ('develop_software', '\\b(?:cod(?:e|ed|ing)|program(?:med|ming)?|implement(?:ed|ing)?|debug(?:ged|ging)?|develop(?:ed|ing)?|maintain(?:ed|ing)?)\\b[^.;|]{0,35}\\b(?:code|software|program|script|pipeline|library|package|platform|application|algorithm)\\b'), ('nominal_software', '\\b(?:code|software|program|script|pipeline|package)\\s+(?:development|implementation|maintenance)\\b')]}


## 12. Safe Rules Learned from Repeated Few-shot Cases

This cell adds conservative rules for frequent phrases that were previously unresolved.

Examples:

- `Acquisition of data` → Investigation
- `Concept and design` → Conceptualization + Methodology
- `compiled the data` → Data Curation
- `revised` → Writing – Review & Editing


In [ ]:
# 12. Safe Rules Learned from Repeated Few-shot Cases

FEW_SHOT_SAFE_PATTERNS_RAW = {'Conceptualization': [('develop_theory', '\\bdevelop(?:ed|ing)?\\b[^.;|]{0,30}\\btheor(?:y|ies)\\b'), ('concept_and_design', '^\\s*(?:concept|conception)\\s+(?:and|&)\\s+design\\s*$')], 'Methodology': [('concept_and_design', '^\\s*(?:concept|conception)\\s+(?:and|&)\\s+design\\s*$'), ('design_only', '^\\s*(?:study\\s+)?design\\s*$'), ('methodologies', '\\bmethodologies\\b')], 'Investigation': [('acquisition_of_data', '\\bacquisition\\s+of\\s+(?:the\\s+)?data\\b'), ('collection_assembly_data', '\\bcollection\\s+and\\s+assembly\\s+of\\s+(?:the\\s+)?data\\b'), ('fabricate_devices', '\\bfabricat(?:e|ed|ing|ion)\\b[^.;|]{0,30}\\b(?:devices?|materials?|electrodes?)\\b'), ('synthesize_materials', '\\bsynthesi[sz](?:e|ed|ing)\\b[^.;|]{0,30}\\b(?:samples?|materials?|compounds?)\\b'), ('laboratory_work', '\\b(?:perform(?:ed|ing)?|conduct(?:ed|ing)?)\\b[^.;|]{0,25}\\blaboratory work\\b'), ('genotyping_sequencing', '\\b(?:genotyping|sequencing|phenotyping|immunohistochemistry)\\b'), ('conducted_study', '^\\s*(?:performed|conducted|carried out)\\s+(?:the\\s+)?study\\s*$')], 'Formal Analysis': [('performed_bioinformatics', '\\b(?:perform(?:ed|ing)?|conduct(?:ed|ing)?)?\\s*bioinformatics(?: analyses?| analysis)?\\b'), ('provided_interpretation', '\\b(?:provid(?:e|ed|ing)|contribut(?:e|ed|ing))\\b[^.;|]{0,30}\\binterpretations?\\b'), ('theoretical_simulation', '\\b(?:calculat(?:e|ed|ing)|perform(?:ed|ing)?)\\b[^.;|]{0,35}\\btheoretical simulations?\\b')], 'Data Curation': [('assembly_of_data', '\\b(?:collection\\s+and\\s+)?assembly\\s+of\\s+(?:the\\s+)?data\\b'), ('compile_collate_data', '\\b(?:compil(?:e|ed|ing)|collat(?:e|ed|ing))\\b[^.;|]{0,25}\\b(?:the\\s+)?data\\b')], 'Writing – Original Draft': [('wrote_only', '^\\s*(?:wrote|drafted|authored)\\s*$'), ('manuscript_draft_only', '^\\s*(?:manuscript\\s+draft|draft\\s+preparation)\\s*$')], 'Writing – Review & Editing': [('revised_edited_only', '^\\s*(?:revised|edited|critically reviewed)\\s*$'), ('reviewing_editing_variant', '\\bwriting\\s*[-–—]?\\s*reviewing\\s+(?:and|&)\\s+editing\\b'), ('commented_on_it', '^\\s*(?:all authors\\s+)?commented\\s+on\\s+it\\s*$')], 'Validation': [('quality_checking', '^\\s*(?:quality[- ]?checking|quality checks?)\\s*$')], 'Project Administration': [('participant_recruitment', '\\b(?:patient|participant|subject)\\s+recruitment\\b|\\brecruit(?:ed|ing)?\\b[^.;|]{0,25}\\b(?:patients?|participants?|subjects?)\\b'), ('project_administrator', '^\\s*project administrator\\s*$'), ('administrative_support', '\\badministrative support\\b')], 'Resources': [('material_support', '\\b(?:technical\\s+or\\s+)?material support\\b')], 'Visualization': [('figures_only', '^\\s*fig(?:ure)?s?\\s*$')]}
for _role, _patterns in FEW_SHOT_SAFE_PATTERNS_RAW.items():
    TEXT_ROLE_PATTERNS_RAW[_role].extend(_patterns)
TEXT_ROLE_PATTERNS = {role: [(name, re.compile(pattern, flags=re.I)) for name, pattern in patterns] for role, patterns in TEXT_ROLE_PATTERNS_RAW.items()}


## 13. Generic Statements, Ambiguity, and Malformed Text

This cell identifies inputs that must not be resolved through similarity alone:

- `variously involved` does not reveal which author performed each role.
- `All authors contributed to this work` contains no specific activity.
- Broken fragments or author names mistakenly extracted as tasks require extraction review.

Such cases are marked as special statements or manual-review cases instead of being forced into a CRediT category.


In [ ]:
# 13. Generic Statements, Ambiguity, and Malformed Text

SPECIAL_TEXT_PATTERNS_RAW = {'approval_statement': [('approval', '\\b(?:read|reviewed|seen|agreed|accepted|gave|given|provided|grant(?:ed)?)\\b[^.;|]{0,70}\\b(?:approv(?:e|ed|al)|final version|publication|published version|submission)\\b'), ('approved_document', '\\bapprov(?:e|ed|al)\\b[^.;|]{0,70}\\b(?:manuscript|paper|article|version|publication|submission)\\b')], 'equal_contribution': [('equal_contribution', '\\b(?:contribut(?:e|ed|ing)\\s+equally|equal contribution|equally contribut(?:e|ed|ing))\\b')], 'collective_role_ambiguity': [('variously_involved', '\\b(?:all authors\\s+)?(?:were|have been|are|being)?\\s*variously involved\\b'), ('variously_contributed', '\\b(?:all authors\\s+)?(?:contributed|participated)\\b[^.;|]{0,80}\\b(?:in varying ways|in different ways|to varying degrees)\\b')], 'accountability_statement': [('accountable', '\\baccountable for all aspects\\b'), ('accuracy_integrity', '\\baccuracy or integrity of any part of the work\\b')], 'generic_contribution': [('generic_single', '^\\s*(?:all authors\\s+)?(?:contributed|participated|assisted|helped|supported|provided input|with input|with support|with contributions)\\s*$'), ('generic_to_work', '^\\s*(?:all authors\\s+)?contributed(?:\\s+(?:extensively|substantially|significantly))?\\s+to\\s+(?:this|the)\\s+(?:work|study|research)\\s*$'), ('help_only', '^\\s*(?:with\\s+(?:the\\s+)?help|with\\s+assistance|with\\s+support)\\s*$'), ('substantial_general', '\\b(?:all authors\\s+)?(?:made|have made)\\s+(?:a\\s+)?substantial contributions? to (?:this|the) work\\b'), ('significant_general', '\\b(?:all authors\\s+)?contributed (?:substantially|significantly) to (?:this|the) work\\b'), ('scientific_discussion_general', '\\b(?:all authors\\s+)?contributed to the scientific discussion\\b'), ('discussion_general', '^\\s*(?:all authors\\s+)?participated in (?:the )?discussion(?:s)?\\s*$'), ('contributed_discussion_general', '^\\s*(?:all authors\\s+)?contributed to (?:the )?discussion(?:s)?\\s*$')], 'malformed_fragment': [('broken_all_authors', '^\\s*(?:all authors\\s*)?[\\(\\[]?[a-z]?\\s*$'), ('short_fragment', '^\\s*[-–—]?[a-z]{1,4}\\s*$'), ('vague_fragment', '^\\s*(?:submission|performed|discussion|supported|helped|assisted)\\s*$'), ('vague_fragment', '^\\s*(?:submission|performed|discussion|supported|helped|assisted)\\s*$'), ('broken_all_authors_qualifier', '^\\s*all authors\\s+(?:except|including|but|at)\\b[^.;|]{0,40}$')]}
SPECIAL_TEXT_PATTERNS = {special_type: [(name, re.compile(pattern, flags=re.I)) for name, pattern in patterns] for special_type, patterns in SPECIAL_TEXT_PATTERNS_RAW.items()}


## 14. Apply Text Rules and Define Context Signals

This cell applies the high-precision text rules and defines context detectors used to prevent false positives.

The context detectors cover manuscript language, literature activity, analysis plans, passive supervision, model training, human training, and participant recruitment.


In [ ]:
# 14. Apply Text Rules and Define Context Signals

def match_high_precision_text_roles(normalized_task: str) -> Tuple[List[str], List[Dict]]:
    """Implement match high precision text roles. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    roles: List[str] = []
    matched: List[Dict] = []
    for role in CREDIT_ROLES:
        for rule_name, pattern in TEXT_ROLE_PATTERNS.get(role, []):
            match = pattern.search(normalized_task)
            if match is None:
                continue
            if role not in roles:
                roles.append(role)
            matched.append({'role': role, 'rule': f'text_{rule_name}', 'evidence': match.group(0), 'span': list(match.span())})
    return apply_contextual_role_corrections(normalized_task, roles, matched)
MANUSCRIPT_CONTEXT_RE = re.compile('\\b(?:manuscript|paper|article|draft|text|submission|version|proofs?)\\b', flags=re.I)
FEEDBACK_CONTEXT_RE = re.compile('\\b(?:feedback|comments?|suggestions?|input|edits?|revisions?)\\b', flags=re.I)
LITERATURE_ACTIVITY_RE = re.compile('(?:\\b(?:systematic\\s+|scoping\\s+|narrative\\s+)?literature\\s+(?:review|search|screening)\\b|\\breview(?:ed|ing)?\\s+(?:the\\s+)?literature\\b)', flags=re.I)
ANALYSIS_PLAN_RE = re.compile('(?:\\b(?:data|statistical|formal|qualitative|quantitative|computational)?\\s*analysis\\s+(?:plan|planning|protocol|strategy|framework)\\b|\\b(?:plan|planning|protocol|strategy|framework)\\s+(?:for|of)\\s+(?:the\\s+)?(?:data\\s+|statistical\\s+|formal\\s+)?analysis\\b)', flags=re.I)
ANALYSIS_EXECUTION_RE = re.compile('\\b(?:analy[sz](?:e|ed|ing)|performed|conducted|ran|run|carried out|implemented|applied|computed|calculated|estimated)\\b', flags=re.I)
PASSIVE_SUPERVISION_RE = re.compile('\\b(?:under\\s+(?:the\\s+)?supervision(?:\\s+of)?|supervised\\s+by)\\b', flags=re.I)
ACTIVE_SUPERVISION_RE = re.compile('\\b(?:supervis(?:e|ed|ing)|oversee|oversaw|overseen|mentor(?:ed|ing)?|guid(?:e|ed|ing)|advis(?:e|ed|ing))\\b', flags=re.I)
MODEL_TRAINING_RE = re.compile('(?:\\b(?:train(?:ed|ing)?|training)\\b[^.;|]{0,35}\\b(?:models?|machine learning|deep learning|neural networks?|classifiers?|algorithms?)\\b|\\b(?:models?|machine learning|deep learning|neural networks?|classifiers?|algorithms?)\\s+training\\b)', flags=re.I)
HUMAN_TRAINING_RE = re.compile('\\btrain(?:ed|ing)?\\b[^.;|]{0,35}\\b(?:students?|researchers?|staff|personnel|trainees?|technicians?|team members?|junior colleagues?)\\b', flags=re.I)
RECRUITMENT_RE = re.compile('(?:\\b(?:patient|participant|subject)\\s+recruitment\\b|\\brecruit(?:ed|ing)?\\b[^.;|]{0,25}\\b(?:patients?|participants?|subjects?)\\b)', flags=re.I)
INVESTIGATION_BEYOND_RECRUITMENT_RE = re.compile('\\b(?:collect(?:ed|ing)?|measure(?:d|ment|ments|ing)?|experiment(?:s|al)?|assays?|interviews?|surveys?|samples?|specimens?|observ(?:e|ed|ation|ations|ing)|data acquisition|data collection)\\b', flags=re.I)
RESULT_DISCUSSION_RULES = {'text_discuss_results', 'text_results_discussion'}
RESULT_DISCUSSION_OPPORTUNITY_RE = re.compile('\\b(?:had|have|was given|were given)\\s+(?:the\\s+)?opportunity\\s+to\\s+discuss\\b', flags=re.I)
RESULTS_SECTION_WRITING_RE = re.compile('\\b(?:wrote|written|write|drafted|prepared|authored)\\b[^.;|]{0,45}\\b(?:results?\\s+and\\s+discussion|discussion)(?:\\s+section)?\\b', flags=re.I)
FEEDBACK_RULES_REQUIRING_MANUSCRIPT = {'text_critical_feedback', 'text_critical_feedback_on_manuscript', 'text_provided_comments', 'text_provided_manuscript_comments', 'provide_manuscript_feedback'}


## 15. Contextual Corrections

This cell corrects common false positives:

- `under the supervision` does not assign Supervision to the person performing the work.
- Training a model is not Supervision.
- Feedback and comments require manuscript context for Writing – Review & Editing.
- An analysis plan belongs to Methodology rather than executed Formal Analysis.
- A literature review belongs to Investigation rather than manuscript editing.
- `discussed the results` is treated as Formal Analysis, except when the phrase only names a written section or describes an opportunity rather than completed work.


In [ ]:
# 15. Contextual Corrections

def apply_contextual_role_corrections(normalized_task: str, roles: Sequence[str], matches: Sequence[Dict]) -> Tuple[List[str], List[Dict]]:
    """Implement apply contextual role corrections. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    task = normalize_task_text(normalized_task)
    has_manuscript = MANUSCRIPT_CONTEXT_RE.search(task) is not None
    has_literature = LITERATURE_ACTIVITY_RE.search(task) is not None
    has_analysis_plan = ANALYSIS_PLAN_RE.search(task) is not None
    has_analysis_execution = ANALYSIS_EXECUTION_RE.search(task) is not None
    has_model_training = MODEL_TRAINING_RE.search(task) is not None
    has_human_training = HUMAN_TRAINING_RE.search(task) is not None
    has_recruitment = RECRUITMENT_RE.search(task) is not None
    recruitment_only = has_recruitment and INVESTIGATION_BEYOND_RECRUITMENT_RE.search(task) is None
    without_passive_supervision = PASSIVE_SUPERVISION_RE.sub(' ', task)
    passive_only_supervision = PASSIVE_SUPERVISION_RE.search(task) is not None and ACTIVE_SUPERVISION_RE.search(without_passive_supervision) is None
    kept_matches: List[Dict] = []
    for item in matches:
        role = item.get('role')
        rule_name = str(item.get('rule', ''))
        if role == 'Formal Analysis':
            if rule_name in RESULT_DISCUSSION_RULES:
                if RESULT_DISCUSSION_OPPORTUNITY_RE.search(task) or RESULTS_SECTION_WRITING_RE.search(task):
                    continue
            if has_analysis_plan and (not has_analysis_execution):
                continue
        if role == 'Supervision':
            if passive_only_supervision:
                continue
            if has_model_training and (not has_human_training):
                continue
        if role == 'Investigation':
            if recruitment_only and rule_name in {'text_collect_evidence', 'collect_or_generate_evidence'}:
                continue
        if role == 'Writing – Review & Editing':
            if has_literature and (not has_manuscript):
                continue
            if rule_name in FEEDBACK_RULES_REQUIRING_MANUSCRIPT and FEEDBACK_CONTEXT_RE.search(task) and (not has_manuscript):
                continue
        kept_matches.append(dict(item))
    corrected_roles = ordered_roles((item.get('role') for item in kept_matches if item.get('role') in CREDIT_ROLES))
    if has_analysis_plan and 'Methodology' not in corrected_roles:
        corrected_roles.append('Methodology')
        kept_matches.append({'role': 'Methodology', 'rule': 'context_analysis_plan', 'evidence': ANALYSIS_PLAN_RE.search(task).group(0)})
    if has_literature and 'Investigation' not in corrected_roles:
        corrected_roles.append('Investigation')
        kept_matches.append({'role': 'Investigation', 'rule': 'context_literature_activity', 'evidence': LITERATURE_ACTIVITY_RE.search(task).group(0)})
    if has_recruitment and 'Project Administration' not in corrected_roles:
        corrected_roles.append('Project Administration')
        kept_matches.append({'role': 'Project Administration', 'rule': 'context_participant_recruitment', 'evidence': RECRUITMENT_RE.search(task).group(0)})
    return (ordered_roles(corrected_roles), kept_matches)

def detect_special_types_v2(normalized_task: str) -> Tuple[List[str], List[Dict]]:
    """Implement detect special types v2. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    special_types: List[str] = []
    matches: List[Dict] = []
    for special_type, patterns in SPECIAL_TEXT_PATTERNS.items():
        for rule_name, pattern in patterns:
            match = pattern.search(normalized_task)
            if match is None:
                continue
            if special_type not in special_types:
                special_types.append(special_type)
            matches.append({'special_type': special_type, 'rule': f'special_{rule_name}', 'evidence': match.group(0), 'span': list(match.span())})
    return (special_types, matches)

def ordered_roles(roles: Iterable[str]) -> List[str]:
    role_set = set(roles)
    return [role for role in CREDIT_ROLES if role in role_set]


## 16. Staged Classification Engine

The decision order is:

1. Explicit CRediT labels.
2. High-precision text rules.
3. Special statements.
4. Dependency-based action–object rules.
5. Nominal phrases.
6. If all prior stages fail: mark the task as `unresolved` and set `credit_send_to_few_shot=True`.

Multiple roles are valid and are not treated as a conflict.


In [ ]:
# 16. Staged Classification Engine

def classify_credit_doc(original_task: str, doc: Doc) -> Dict:
    """Implement classify credit doc. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    normalized_task = normalize_task_text(original_task)
    result = {'original_task': original_task, 'normalized_task': normalized_task, 'roles': [], 'primary_role': None, 'assignment_method': None, 'matched_rules': [], 'confidence': 0.0, 'needs_split': False, 'contains_multiple_roles': False, 'special_type': None, 'special_types': [], 'send_to_few_shot': False, 'needs_manual_review': False, 'candidate_roles': [], 'ambiguous_predicates': []}
    if not normalized_task:
        result['assignment_method'] = 'empty'
        return result
    if normalized_task in STANDALONE_NOISE:
        result['assignment_method'] = 'standalone_noise'
        result['special_type'] = 'noise'
        result['special_types'] = ['noise']
        result['confidence'] = 1.0
        return result
    explicit_roles = detect_explicit_roles(normalized_task)
    if explicit_roles:
        explicit_roles = ordered_roles(explicit_roles)
        result['roles'] = explicit_roles
        result['primary_role'] = explicit_roles[0] if len(explicit_roles) == 1 else None
        result['assignment_method'] = 'explicit_role' if len(explicit_roles) == 1 else 'explicit_multi_role'
        result['matched_rules'] = [{'role': role, 'rule': 'explicit_credit_label', 'evidence': normalized_task} for role in explicit_roles]
        result['confidence'] = 1.0
        result['contains_multiple_roles'] = len(explicit_roles) > 1
        return result
    text_roles, text_matches = match_high_precision_text_roles(normalized_task)
    special_types, special_matches = detect_special_types_v2(normalized_task)
    blocking_special_types = {'collective_role_ambiguity'}
    if blocking_special_types.intersection(special_types):
        result['assignment_method'] = 'special_rule'
        result['special_types'] = special_types
        result['special_type'] = 'collective_role_ambiguity'
        result['matched_rules'] = special_matches
        result['confidence'] = 1.0
        result['needs_manual_review'] = True
        result['send_to_few_shot'] = False
        return result
    if text_roles:
        text_roles = ordered_roles(text_roles)
        result['roles'] = text_roles
        result['primary_role'] = text_roles[0] if len(text_roles) == 1 else None
        result['assignment_method'] = 'text_rule' if len(text_roles) == 1 else 'text_multi_role'
        result['matched_rules'] = text_matches + special_matches
        result['confidence'] = 0.98
        result['contains_multiple_roles'] = len(text_roles) > 1
        result['special_types'] = special_types
        result['special_type'] = special_types[0] if special_types else None
        return result
    if special_types:
        result['assignment_method'] = 'special_rule'
        result['special_types'] = special_types
        result['special_type'] = special_types[0]
        result['matched_rules'] = special_matches
        result['confidence'] = 1.0
        result['needs_manual_review'] = any((special_type in {'malformed_fragment', 'collective_role_ambiguity'} for special_type in special_types))
        return result
    old_special_type = detect_special_type(normalized_task)
    if old_special_type is not None:
        result['assignment_method'] = 'special_rule'
        result['special_type'] = old_special_type
        result['special_types'] = [old_special_type]
        result['confidence'] = 1.0
        return result
    predicate_role_map: Dict[int, Set[str]] = defaultdict(set)
    predicate_matches: Dict[int, List[Dict]] = defaultdict(list)
    for verb in get_action_tokens(doc):
        context = get_predicate_context(verb)
        participation_roles, participation_matches = classify_participation_predicate(context)
        for role in participation_roles:
            predicate_role_map[verb.i].add(role)
        for item in participation_matches:
            predicate_matches[verb.i].append({**item, 'predicate': verb.text, 'predicate_index': verb.i, 'evidence': context['evidence_text']})
        for current_rule in ACTION_TARGET_RULES:
            if not match_action_target_rule(context, current_rule):
                continue
            predicate_role_map[verb.i].add(current_rule.role)
            predicate_matches[verb.i].append({'role': current_rule.role, 'rule': current_rule.name, 'predicate': verb.text, 'predicate_lemma': context['action'], 'predicate_index': verb.i, 'target_text': context['target_text'], 'evidence': context['evidence_text'], 'used_shared_target': context['used_shared_target']})
    dependency_matches_flat = [item for predicate_index in sorted(predicate_matches) for item in predicate_matches[predicate_index]]
    dependency_roles, dependency_matches_flat = apply_contextual_role_corrections(normalized_task, ordered_roles((role for roles in predicate_role_map.values() for role in roles)), dependency_matches_flat)
    if dependency_roles:
        result['roles'] = dependency_roles
        result['primary_role'] = dependency_roles[0] if len(dependency_roles) == 1 else None
        result['assignment_method'] = 'dependency_rule' if len(dependency_roles) == 1 else 'dependency_multi_role'
        result['matched_rules'] = dependency_matches_flat
        result['confidence'] = 0.95
        result['contains_multiple_roles'] = len(dependency_roles) > 1
        result['ambiguous_predicates'] = [{'predicate_index': predicate_index, 'roles': ordered_roles(roles), 'matches': predicate_matches[predicate_index]} for predicate_index, roles in predicate_role_map.items() if len(roles) > 1]
        return result
    nominal_roles, nominal_matches = match_nominal_roles(normalized_task)
    if nominal_roles:
        nominal_roles, nominal_matches = apply_contextual_role_corrections(normalized_task, ordered_roles(nominal_roles), nominal_matches)
    if nominal_roles:
        result['roles'] = nominal_roles
        result['primary_role'] = nominal_roles[0] if len(nominal_roles) == 1 else None
        result['assignment_method'] = 'nominal_rule' if len(nominal_roles) == 1 else 'nominal_multi_role'
        result['matched_rules'] = nominal_matches
        result['confidence'] = 0.93
        result['contains_multiple_roles'] = len(nominal_roles) > 1
        return result
    result['assignment_method'] = 'unresolved'
    result['candidate_roles'] = generate_candidate_roles(normalized_task)
    result['send_to_few_shot'] = True
    return result

def classify_credit_task(task) -> Dict:
    """Implement classify credit task. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    original_task = safe_text(task)
    doc = nlp(original_task)
    return classify_credit_doc(original_task, doc)


## 17. Prepare Tasks and Create Task-level Records

The `tasks` column is split only on a single `|` delimiter. A scientific expression containing `||` is preserved.

Each unique task is classified once, and the result is reused for every occurrence. The output contains one row per author–task pair with roles, matched rules, confidence, special types, and the few-shot flag.


In [ ]:
# 17. Prepare Tasks and Create Task-level Records

def split_pipe_tasks(value) -> List[str]:
    """Implement split pipe tasks. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_tasks = list(value)
    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass
        raw_tasks = re.split('(?<!\\|)\\|(?!\\|)', str(value))
    tasks = []
    for raw_task in raw_tasks:
        task = safe_text(raw_task).strip()
        if task:
            tasks.append(task)
    return tasks

def classify_unique_tasks(tasks: Sequence[str], batch_size: int=256) -> Dict[str, Dict]:
    """Implement classify unique tasks. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    unique_tasks = list(dict.fromkeys((task for task in tasks if task)))
    results: Dict[str, Dict] = {}
    if not unique_tasks:
        return results
    for task, doc in zip(unique_tasks, nlp.pipe(unique_tasks, batch_size=batch_size)):
        results[task] = classify_credit_doc(original_task=task, doc=doc)
    return results

def get_all_row_roles(task_results: Sequence[Dict]) -> List[str]:
    """Implement get all row roles. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    found_roles = {role for result in task_results for role in result['roles']}
    return [role for role in CREDIT_ROLES if role in found_roles]

def make_task_level_record(base_row: Dict, task: str, task_position: int, result: Dict) -> Dict:
    """Implement make task level record. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    record = dict(base_row)
    record['task_position'] = task_position
    record['task_single'] = task
    record['credit_roles_rule'] = json.dumps(result['roles'], ensure_ascii=False)
    record['credit_primary_role'] = result['primary_role']
    record['credit_assignment_method'] = result['assignment_method']
    record['credit_matched_rules'] = json.dumps(result['matched_rules'], ensure_ascii=False)
    record['credit_rule_confidence'] = result['confidence']
    record['credit_needs_split'] = result['needs_split']
    record['credit_contains_multiple_roles'] = result.get('contains_multiple_roles', len(result.get('roles', [])) > 1)
    record['credit_special_type'] = result['special_type']
    record['credit_special_types'] = json.dumps(result.get('special_types', []), ensure_ascii=False)
    record['credit_needs_manual_review'] = result.get('needs_manual_review', False)
    record['credit_send_to_few_shot'] = result['send_to_few_shot']
    record['credit_candidate_roles'] = json.dumps(result['candidate_roles'], ensure_ascii=False)
    record['credit_ambiguous_predicates'] = json.dumps(result['ambiguous_predicates'], ensure_ascii=False)
    record['credit_normalized_task'] = result['normalized_task']
    return record


## 18. Apply the Rule Engine to the Entire DataFrame

This cell runs the rule engine over all authors and returns:

- `mapping_credit_df`: one row per author.
- `tasks_credit_df`: one row per author–task pair.

The second DataFrame is used by the embedding stage.


In [ ]:
# 18. Apply the Rule Engine to the Entire DataFrame

def apply_credit_rules_to_mapping_df(mapping_df: pd.DataFrame, tasks_column: str='tasks', batch_size: int=256) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Implement apply credit rules to mapping df. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    if tasks_column not in mapping_df.columns:
        raise KeyError(f'See the preceding markdown cell for details.{tasks_column!r}See the preceding markdown cell for details.{list(mapping_df.columns)}')
    working_df = mapping_df.copy()
    if '_mapping_row_id' in working_df.columns:
        working_df = working_df.drop(columns=['_mapping_row_id'])
    working_df.insert(0, '_mapping_row_id', range(len(working_df)))
    task_lists = working_df[tasks_column].apply(split_pipe_tasks).tolist()
    all_task_instances = [task for row_tasks in task_lists for task in row_tasks]
    unique_task_count = len(set(all_task_instances))
    print('=' * 100)
    print('Running the CRediT rule engine')
    print('=' * 100)
    print(f'Number of author rows: {len(working_df):,}')
    print(f'Number of task instances: {len(all_task_instances):,}')
    print(f'Number of unique tasks: {unique_task_count:,}')
    result_lookup = classify_unique_tasks(tasks=all_task_instances, batch_size=batch_size)
    row_task_results = [[result_lookup[task] for task in row_tasks] for row_tasks in task_lists]
    mapping_credit_df = working_df.copy()
    mapping_credit_df['tasks_parsed'] = [json.dumps(row_tasks, ensure_ascii=False) for row_tasks in task_lists]
    mapping_credit_df['credit_task_count'] = [len(row_tasks) for row_tasks in task_lists]
    mapping_credit_df['credit_results_per_task'] = [json.dumps(task_results, ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_roles_per_task'] = [json.dumps([result['roles'] for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_primary_role_per_task'] = [json.dumps([result['primary_role'] for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_method_per_task'] = [json.dumps([result['assignment_method'] for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_needs_split_per_task'] = [json.dumps([result['needs_split'] for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_contains_multiple_roles_per_task'] = [json.dumps([result.get('contains_multiple_roles', len(result.get('roles', [])) > 1) for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_special_types_per_task'] = [json.dumps([result.get('special_types', []) for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_needs_manual_review_per_task'] = [json.dumps([result.get('needs_manual_review', False) for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_send_to_few_shot_per_task'] = [json.dumps([result['send_to_few_shot'] for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_candidate_roles_per_task'] = [json.dumps([result['candidate_roles'] for result in task_results], ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_any_send_to_few_shot'] = [any((result['send_to_few_shot'] for result in task_results)) for task_results in row_task_results]
    mapping_credit_df['credit_any_needs_split'] = [any((result['needs_split'] for result in task_results)) for task_results in row_task_results]
    mapping_credit_df['credit_any_multiple_roles'] = [any((result.get('contains_multiple_roles', len(result.get('roles', [])) > 1) for result in task_results)) for task_results in row_task_results]
    mapping_credit_df['credit_any_manual_review'] = [any((result.get('needs_manual_review', False) for result in task_results)) for task_results in row_task_results]
    mapping_credit_df['credit_all_roles'] = [json.dumps(get_all_row_roles(task_results), ensure_ascii=False) for task_results in row_task_results]
    mapping_credit_df['credit_few_shot_task_count'] = [sum((result['send_to_few_shot'] for result in task_results)) for task_results in row_task_results]
    mapping_credit_df['credit_needs_split_task_count'] = [sum((result['needs_split'] for result in task_results)) for task_results in row_task_results]
    task_level_records = []
    for row_position, (_, row) in enumerate(working_df.iterrows()):
        base_row = row.to_dict()
        row_tasks = task_lists[row_position]
        task_results = row_task_results[row_position]
        for task_position, (task, result) in enumerate(zip(row_tasks, task_results), start=1):
            record = make_task_level_record(base_row=base_row, task=task, task_position=task_position, result=result)
            task_level_records.append(record)
    tasks_credit_df = pd.DataFrame(task_level_records)
    print('\n' + '=' * 100)
    print('Results summary')
    print('=' * 100)
    print(f'Number of rows in the task-level DataFrame: {len(tasks_credit_df):,}')
    if not tasks_credit_df.empty:
        print('\nAssignment method:')
        print(tasks_credit_df['credit_assignment_method'].value_counts(dropna=False))
        print('\nRequires few-shot:')
        print(tasks_credit_df['credit_send_to_few_shot'].value_counts(dropna=False))
        print('\nContains multiple CRediT roles:')
        print(tasks_credit_df['credit_contains_multiple_roles'].value_counts(dropna=False))
        print('\nPrimary role:')
        print(tasks_credit_df['credit_primary_role'].value_counts(dropna=False))
    return (mapping_credit_df, tasks_credit_df)


## 19. Pipeline Wrapper

This short function provides one entry point for running the entire deterministic rule stage.


In [ ]:
# 19. Pipeline Wrapper

def run_credit_pipeline(mapping_df_nature: pd.DataFrame, tasks_column: str='tasks', batch_size: int=256) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Implement run credit pipeline. See the preceding markdown cell for the complete purpose, rules, examples, and edge cases."""
    mapping_credit_df, tasks_credit_df = apply_credit_rules_to_mapping_df(mapping_df=mapping_df_nature, tasks_column=tasks_column, batch_size=batch_size)
    return (mapping_credit_df, tasks_credit_df)


## 19A. Preserve Multiple Actions Inside One Contribution Task

The original task text is never deleted or globally split.

This layer creates an internal action-level representation only when the structure provides strong evidence that several actions are present.

### Governed activity lists

`contributed to CNN training data generation, manuscript text, and response to reviewers`

is represented internally as:

1. `contributed to CNN training data generation`
2. `contributed to manuscript text`
3. `contributed to response to reviewers`

The three actions can therefore receive different CRediT roles.

### Coordinated verbs with a shared object

`collected and analyzed the data`

is represented as:

1. `collected the data`
2. `analyzed the data`

### Repeated roles are preserved

Under the requested project policy:

`prepared and revised the manuscript`

is represented as two separate actions:

1. `prepared the manuscript` → Writing – Review & Editing
2. `revised the manuscript` → Writing – Review & Editing

The task-level role list contains unique categories, while the detailed action mapping preserves both actions even when they receive the same category.


In [ ]:
import ast

PARTICIPATION_LIST_RE = re.compile(
    r"^(?P<prefix>.*?\b(?:contribut(?:e|ed|ing)|participat(?:e|ed|ing)|"
    r"assist(?:ed|ing)?|help(?:ed|ing)?|support(?:ed|ing)?)\s+"
    r"(?P<prep>to|in|with)\s+)(?P<body>.+)$",
    flags=re.I,
)

ROLE_BEARING_FRAGMENT_RE = re.compile(
    r"\b(?:concept|design|method|protocol|experiment|assay|data|sample|"
    r"analysis|interpret|model|software|code|fund|grant|resource|material|"
    r"manuscript|paper|article|draft|text|reviewer|revision|editing|"
    r"figure|graph|plot|table|supervis|mentor|validation|recruitment)\b",
    flags=re.I,
)

PREPARE_REVISE_DOCUMENT_RE = re.compile(
    r"\b(?P<first>prepar(?:e|ed|ing))\s+(?:and|&)\s+"
    r"(?P<second>revis(?:e|ed|ing))\s+"
    r"(?P<object>(?:the\s+)?(?:manuscript|paper|article|draft|text))\b",
    flags=re.I,
)

REVISE_PREPARE_DOCUMENT_RE = re.compile(
    r"\b(?P<first>revis(?:e|ed|ing))\s+(?:and|&)\s+"
    r"(?P<second>prepar(?:e|ed|ing))\s+"
    r"(?P<object>(?:the\s+)?(?:manuscript|paper|article|draft|text))\b",
    flags=re.I,
)

ATOMIC_ROLE_PATTERNS = [
    (
        re.compile(r"\b(?:cnn\s+)?training data generation\b", flags=re.I),
        ["Investigation"],
        "atomic_training_data_generation",
    ),
    (
        re.compile(
            r"\b(?:contribut(?:e|ed|ing)\s+to\s+)?"
            r"(?:the\s+)?(?:manuscript|paper|article)\s+text\b",
            flags=re.I,
        ),
        ["Writing – Original Draft"],
        "atomic_manuscript_text",
    ),
    (
        re.compile(
            r"\b(?:response|responses|reply|replies)\s+to\s+reviewers?\b",
            flags=re.I,
        ),
        ["Writing – Review & Editing"],
        "atomic_response_to_reviewers",
    ),
]


def parse_list_value(value):
    """Convert a real list or serialized list into a clean Python list."""
    if isinstance(value, (list, tuple, set)):
        return [safe_text(item).strip() for item in value if safe_text(item).strip()]

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = safe_text(value).strip()
    if not text:
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, (list, tuple, set)):
                return [
                    safe_text(item).strip()
                    for item in parsed
                    if safe_text(item).strip()
                ]
        except Exception:
            pass

    return [text]


def split_enumerated_body(body):
    """
    Split only a list governed by a participation verb.

    This does not split every comma or every 'and' in the dataset.
    """
    body = re.sub(r"\s+", " ", safe_text(body)).strip(" ,;")
    if not body:
        return []

    def clean_part(part):
        part = part.strip(" ,;")
        part = re.sub(
            r"^(?:and|as well as)\s+",
            "",
            part,
            flags=re.I,
        )
        return part.strip(" ,;")

    comma_parts = [
        clean_part(part)
        for part in body.split(",")
        if clean_part(part)
    ]

    if len(comma_parts) > 1:
        final_part = comma_parts.pop()
        comma_parts.extend(
            clean_part(part)
            for part in re.split(
                r"\s+(?:and|as well as)\s+",
                final_part,
                maxsplit=1,
                flags=re.I,
            )
            if clean_part(part)
        )
        return comma_parts

    and_parts = [
        clean_part(part)
        for part in re.split(
            r"\s+(?:and|as well as)\s+",
            body,
            maxsplit=1,
            flags=re.I,
        )
        if clean_part(part)
    ]

    if (
        len(and_parts) == 2
        and all(ROLE_BEARING_FRAGMENT_RE.search(part) for part in and_parts)
    ):
        return and_parts

    return [body]


def _token_text(tokens):
    """Create readable text from an ordered token sequence."""
    text = " ".join(
        token.text
        for token in tokens
        if not token.is_space and not token.is_punct
    )
    return re.sub(r"\s+", " ", text).strip()


def extract_atomic_action_specs(original_task, doc):
    """
    Extract internal atomic actions while preserving the original task.

    The function is conservative: it splits governed activity lists and
    coordinated verbs with a shared object, but it does not globally split
    all commas or conjunctions.
    """
    task = safe_text(original_task).strip()
    specs = []

    participation_match = PARTICIPATION_LIST_RE.match(task)
    if participation_match:
        prefix = participation_match.group("prefix").strip()
        body_parts = split_enumerated_body(participation_match.group("body"))

        if len(body_parts) > 1:
            for part in body_parts:
                specs.append({
                    "action_text": f"{prefix} {part}".strip(),
                    "source": "participation_list",
                    "forced_roles": [],
                })

    for pattern in (
        PREPARE_REVISE_DOCUMENT_RE,
        REVISE_PREPARE_DOCUMENT_RE,
    ):
        match = pattern.search(task)
        if match:
            shared_object = match.group("object")
            specs.extend([
                {
                    "action_text": f"{match.group('first')} {shared_object}",
                    "source": "shared_document_object",
                    "forced_roles": ["Writing – Review & Editing"],
                },
                {
                    "action_text": f"{match.group('second')} {shared_object}",
                    "source": "shared_document_object",
                    "forced_roles": ["Writing – Review & Editing"],
                },
            ])

    if doc.has_annotation("DEP"):
        seen_groups = set()

        for verb in get_action_tokens(doc):
            group = get_conj_verb_group(verb)
            group_key = tuple(token.i for token in group)

            if len(group) < 2 or group_key in seen_groups:
                continue

            seen_groups.add(group_key)

            for coordinated_verb in group:
                target_tokens, _ = get_effective_target_tokens(coordinated_verb)
                target_text = _token_text(target_tokens)

                if not target_text:
                    continue

                specs.append({
                    "action_text": (
                        f"{coordinated_verb.text} {target_text}"
                    ).strip(),
                    "source": "coordinated_verbs",
                    "forced_roles": [],
                })

    if not specs:
        specs = [{
            "action_text": task,
            "source": "original_task",
            "forced_roles": [],
        }]

    merged = {}
    order = []

    for spec in specs:
        key = normalize_task_text(spec["action_text"])
        if not key:
            continue

        if key not in merged:
            merged[key] = {
                "action_text": spec["action_text"],
                "source": spec["source"],
                "forced_roles": list(spec["forced_roles"]),
            }
            order.append(key)
        else:
            merged[key]["forced_roles"] = ordered_roles(
                merged[key]["forced_roles"] + spec["forced_roles"]
            )

    return [merged[key] for key in order]


def apply_atomic_role_overrides(action_text, roles, matched_rules):
    """Add only narrowly defined action-level roles."""
    final_roles = list(roles)
    final_matches = list(matched_rules)

    for pattern, extra_roles, rule_name in ATOMIC_ROLE_PATTERNS:
        match = pattern.search(action_text)
        if match is None:
            continue

        for role in extra_roles:
            if role not in final_roles:
                final_roles.append(role)

            final_matches.append({
                "role": role,
                "rule": rule_name,
                "evidence": match.group(0),
            })

    return ordered_roles(final_roles), final_matches


def _base_roles_after_project_policy(original_task, base_result):
    """
    Apply the requested policy for 'prepared and revised the manuscript'.

    Under this policy, both atomic actions are treated as
    Writing – Review & Editing, so the broad Original Draft match is removed.
    """
    roles = list(base_result.get("roles", []))

    if (
        PREPARE_REVISE_DOCUMENT_RE.search(original_task)
        or REVISE_PREPARE_DOCUMENT_RE.search(original_task)
    ):
        roles = [
            role
            for role in roles
            if role != "Writing – Original Draft"
        ]

    return ordered_roles(roles)


def build_action_aware_lookup(tasks, batch_size=256):
    """
    Build one enhanced result per unique original task.

    Every result contains both:
    - unique task-level roles;
    - a detailed list of atomic actions, preserving repeated roles.
    """
    unique_tasks = list(dict.fromkeys(task for task in tasks if task))
    if not unique_tasks:
        return {}

    original_docs = list(
        nlp.pipe(unique_tasks, batch_size=batch_size)
    )

    base_lookup = {}
    specs_lookup = {}
    all_atomic_texts = []

    for task, doc in zip(unique_tasks, original_docs):
        base_lookup[task] = classify_credit_doc(task, doc)
        specs = extract_atomic_action_specs(task, doc)
        specs_lookup[task] = specs
        all_atomic_texts.extend(spec["action_text"] for spec in specs)

    atomic_result_lookup = classify_unique_tasks(
        all_atomic_texts,
        batch_size=batch_size,
    )

    enhanced_lookup = {}

    for task in unique_tasks:
        base_result = dict(base_lookup[task])
        base_roles = _base_roles_after_project_policy(task, base_result)

        action_mappings = []
        combined_roles = list(base_roles)
        unresolved_actions = []
        candidate_roles = list(base_result.get("candidate_roles", []))

        for action_index, spec in enumerate(
            specs_lookup[task],
            start=1,
        ):
            action_text = spec["action_text"]
            action_result = dict(atomic_result_lookup[action_text])

            original_action_roles = list(
                action_result.get("roles", [])
            )

            action_roles, action_matches = apply_atomic_role_overrides(
                action_text,
                original_action_roles,
                action_result.get("matched_rules", []),
            )

            assignment_method = action_result.get(
                "assignment_method"
            )

            if action_roles and not original_action_roles:
                assignment_method = "atomic_text_rule"

            if spec["forced_roles"]:
                action_roles = ordered_roles(spec["forced_roles"])
                assignment_method = "atomic_shared_object_rule"
                action_matches = [
                    {
                        "role": role,
                        "rule": "shared_document_revision_policy",
                        "evidence": action_text,
                    }
                    for role in action_roles
                ]

            action_send_to_few_shot = (
                len(action_roles) == 0
                and bool(action_result.get("send_to_few_shot", False))
            )

            if action_send_to_few_shot:
                unresolved_actions.append(action_text)

            combined_roles.extend(action_roles)
            candidate_roles.extend(
                action_result.get("candidate_roles", [])
            )

            action_mappings.append({
                "action_index": action_index,
                "action_text": action_text,
                "source": spec["source"],
                "roles_rule": action_roles,
                "assignment_method_rule": assignment_method,
                "matched_rules": action_matches,
                "send_to_few_shot": action_send_to_few_shot,
                "candidate_roles": action_result.get(
                    "candidate_roles", []
                ),
                "special_type": action_result.get("special_type"),
                "needs_manual_review": action_result.get(
                    "needs_manual_review", False
                ),
            })

        combined_roles = ordered_roles(combined_roles)

        enhanced_lookup[task] = {
            **base_result,
            "roles": combined_roles,
            "primary_role": (
                combined_roles[0]
                if len(combined_roles) == 1
                else None
            ),
            "assignment_method": (
                "action_aware_multi_role"
                if len(combined_roles) > 1
                else (
                    "action_aware_rule"
                    if combined_roles
                    else base_result.get("assignment_method")
                )
            ),
            "contains_multiple_roles": len(combined_roles) > 1,
            "send_to_few_shot": len(unresolved_actions) > 0,
            "candidate_roles": ordered_roles(candidate_roles),
            "action_mappings_rule": action_mappings,
            "unresolved_actions_rule": unresolved_actions,
            "atomic_action_count": len(action_mappings),
        }

    return enhanced_lookup


def create_action_level_dataframe(tasks_credit_df, action_lookup):
    """Create one row per original task and atomic action."""
    records = []

    for _, row in tasks_credit_df.iterrows():
        enhanced = action_lookup[row["task_single"]]

        for action in enhanced["action_mappings_rule"]:
            record = row.to_dict()
            record.update({
                "action_index": action["action_index"],
                "action_text": action["action_text"],
                "action_source": action["source"],
                "credit_roles_rule": json.dumps(
                    action["roles_rule"],
                    ensure_ascii=False,
                ),
                "credit_assignment_method": action[
                    "assignment_method_rule"
                ],
                "credit_matched_rules": json.dumps(
                    action["matched_rules"],
                    ensure_ascii=False,
                ),
                "credit_send_to_few_shot": action[
                    "send_to_few_shot"
                ],
                "credit_candidate_roles": json.dumps(
                    action["candidate_roles"],
                    ensure_ascii=False,
                ),
                "credit_special_type": action["special_type"],
                "credit_needs_manual_review": action[
                    "needs_manual_review"
                ],
            })
            records.append(record)

    return pd.DataFrame(records)


def apply_action_aware_layer(
    mapping_credit_df,
    tasks_credit_df,
    batch_size=256,
):
    """
    Enhance existing rule outputs with action-level coverage.

    The original task remains intact. The function adds detailed action
    mappings and updates task-level role unions and unresolved flags.
    """
    unique_tasks = (
        tasks_credit_df["task_single"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    action_lookup = build_action_aware_lookup(
        unique_tasks,
        batch_size=batch_size,
    )

    enhanced_task_df = tasks_credit_df.copy()

    enhanced_task_df["credit_roles_rule"] = enhanced_task_df[
        "task_single"
    ].map(
        lambda task: json.dumps(
            action_lookup[task]["roles"],
            ensure_ascii=False,
        )
    )
    enhanced_task_df["credit_primary_role"] = enhanced_task_df[
        "task_single"
    ].map(lambda task: action_lookup[task]["primary_role"])
    enhanced_task_df["credit_assignment_method"] = enhanced_task_df[
        "task_single"
    ].map(lambda task: action_lookup[task]["assignment_method"])
    enhanced_task_df["credit_contains_multiple_roles"] = enhanced_task_df[
        "task_single"
    ].map(
        lambda task: action_lookup[task][
            "contains_multiple_roles"
        ]
    )
    enhanced_task_df["credit_send_to_few_shot"] = enhanced_task_df[
        "task_single"
    ].map(
        lambda task: action_lookup[task]["send_to_few_shot"]
    )
    enhanced_task_df["credit_candidate_roles"] = enhanced_task_df[
        "task_single"
    ].map(
        lambda task: json.dumps(
            action_lookup[task]["candidate_roles"],
            ensure_ascii=False,
        )
    )
    enhanced_task_df["credit_action_mappings_rule"] = enhanced_task_df[
        "task_single"
    ].map(
        lambda task: json.dumps(
            action_lookup[task]["action_mappings_rule"],
            ensure_ascii=False,
        )
    )
    enhanced_task_df["credit_unresolved_actions_rule"] = enhanced_task_df[
        "task_single"
    ].map(
        lambda task: json.dumps(
            action_lookup[task]["unresolved_actions_rule"],
            ensure_ascii=False,
        )
    )
    enhanced_task_df["credit_atomic_action_count"] = enhanced_task_df[
        "task_single"
    ].map(
        lambda task: action_lookup[task]["atomic_action_count"]
    )

    actions_credit_df = create_action_level_dataframe(
        enhanced_task_df,
        action_lookup,
    )

    enhanced_mapping_df = mapping_credit_df.copy()

    grouped = (
        enhanced_task_df.sort_values(
            ["_mapping_row_id", "task_position"]
        )
        .groupby("_mapping_row_id", sort=False)
    )

    task_roles_by_row = grouped["credit_roles_rule"].apply(list)
    task_methods_by_row = grouped[
        "credit_assignment_method"
    ].apply(list)
    task_few_shot_by_row = grouped[
        "credit_send_to_few_shot"
    ].apply(list)
    task_actions_by_row = grouped[
        "credit_action_mappings_rule"
    ].apply(list)

    enhanced_mapping_df[
        "credit_roles_per_task"
    ] = enhanced_mapping_df["_mapping_row_id"].map(
        lambda row_id: json.dumps(
            [
                json.loads(value)
                for value in task_roles_by_row.get(row_id, [])
            ],
            ensure_ascii=False,
        )
    )
    enhanced_mapping_df[
        "credit_method_per_task"
    ] = enhanced_mapping_df["_mapping_row_id"].map(
        lambda row_id: json.dumps(
            task_methods_by_row.get(row_id, []),
            ensure_ascii=False,
        )
    )
    enhanced_mapping_df[
        "credit_send_to_few_shot_per_task"
    ] = enhanced_mapping_df["_mapping_row_id"].map(
        lambda row_id: json.dumps(
            task_few_shot_by_row.get(row_id, []),
            ensure_ascii=False,
        )
    )
    enhanced_mapping_df[
        "credit_action_mappings_per_task"
    ] = enhanced_mapping_df["_mapping_row_id"].map(
        lambda row_id: json.dumps(
            [
                json.loads(value)
                for value in task_actions_by_row.get(row_id, [])
            ],
            ensure_ascii=False,
        )
    )
    enhanced_mapping_df[
        "credit_any_send_to_few_shot"
    ] = enhanced_mapping_df["_mapping_row_id"].map(
        lambda row_id: any(
            task_few_shot_by_row.get(row_id, [])
        )
    )
    enhanced_mapping_df[
        "credit_few_shot_task_count"
    ] = enhanced_mapping_df["_mapping_row_id"].map(
        lambda row_id: sum(
            task_few_shot_by_row.get(row_id, [])
        )
    )
    enhanced_mapping_df[
        "credit_all_roles"
    ] = enhanced_mapping_df["_mapping_row_id"].map(
        lambda row_id: json.dumps(
            ordered_roles(
                role
                for value in task_roles_by_row.get(row_id, [])
                for role in json.loads(value)
            ),
            ensure_ascii=False,
        )
    )

    return (
        enhanced_mapping_df,
        enhanced_task_df,
        actions_credit_df,
    )


def run_credit_pipeline_action_aware(
    mapping_df,
    tasks_column="tasks",
    batch_size=256,
):
    """Run the deterministic pipeline and then add action-level coverage."""
    mapping_credit_df, tasks_credit_df = run_credit_pipeline(
        mapping_df,
        tasks_column=tasks_column,
        batch_size=batch_size,
    )

    return apply_action_aware_layer(
        mapping_credit_df,
        tasks_credit_df,
        batch_size=batch_size,
    )


## 20. Load the Mapping File

Set the path to the file containing the `tasks` column.

The loader supports CSV and Excel. If `mapping_df_nature` already exists in memory, the notebook does not reload it.


In [ ]:
# 20. Load the Mapping File

from pathlib import Path
MAPPING_FILE_PATH = '/content/nature_author_task_mapping_all_articles_unique.csv'
TASKS_COLUMN = 'tasks'

def load_mapping_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    return pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
if 'mapping_df_nature' not in globals():
    mapping_df_nature = load_mapping_file(MAPPING_FILE_PATH)
print('Rows:', len(mapping_df_nature))
print('Columns:', list(mapping_df_nature.columns))
display(mapping_df_nature.head(3))


Rows: 195136
Columns: ['dataset', 'source_folder', 'file_name', 'file_path', 'source_key', 'local_article_index', 'global_article_index', 'article_instance_key', 'title', 'year', 'url', 'full_name', 'author_index', 'full_name_indices', 'tasks', 'n_tasks', 'matched_aliases', 'alias_rules', 'raw_clauses', 'resolution_statuses', 'task_directions', 'contains_respectively', 'needs_review', 'postprocess_unresolved_aliases', 'postprocess_rules', 'postprocess_confidence', 'duplicate_group_size', 'duplicate_instances_removed', 'merged_article_instance_keys', 'merged_source_folders', 'merged_file_names', 'deduplication_status', 'information_loss_check']


,dataset,source_folder,file_name,file_path,source_key,local_article_index,global_article_index,article_instance_key,title,year,...,postprocess_unresolved_aliases,postprocess_rules,postprocess_confidence,duplicate_group_size,duplicate_instances_removed,merged_article_instance_keys,merged_source_folders,merged_file_names,deduplication_status,information_loss_check
0,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,2024,...,NaN,unchanged,high,1,0,nature|| humanities and social sciences commun...,nature,humanities and social sciences communications...,unique_article,not_applicable
1,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,2024,...,NaN,unchanged,high,1,0,nature|| humanities and social sciences commun...,nature,humanities and social sciences communications...,unique_article,not_applicable
2,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,2024,...,NaN,unchanged,high,1,0,nature|| humanities and social sciences commun...,nature,humanities and social sciences communications...,unique_article,not_applicable


## 21. Run the Rule and Action-aware Stages

This cell first runs the deterministic CRediT engine and then applies the action-aware layer.

It returns:

- `mapping_df_nature_credit`: one row per author.
- `mapping_df_nature_tasks_credit`: one row per original author–task pair.
- `mapping_df_nature_actions_credit`: one row per atomic action.

Only unresolved atomic actions continue to the few-shot embedding stage.


In [ ]:
mapping_df_nature_credit, mapping_df_nature_tasks_credit, mapping_df_nature_actions_credit = (
    run_credit_pipeline_action_aware(
        mapping_df_nature,
        tasks_column=TASKS_COLUMN,
        batch_size=256,
    )
)


Running the CRediT rule engine
Number of author rows: 195,136
Number of task instances: 405,505
Number of unique tasks: 81,375

Results summary
Number of rows in the task-level DataFrame: 405,505

Assignment method:
credit_assignment_method
text_rule                200720
text_multi_role           83255
unresolved                47799
explicit_role             32172
special_rule              28990
dependency_rule            7661
explicit_multi_role        4260
nominal_rule                367
dependency_multi_role       221
standalone_noise             60
Name: count, dtype: int64

Requires few-shot:
credit_send_to_few_shot
False    357706
True      47799
Name: count, dtype: int64

Contains multiple CRediT roles:
credit_contains_multiple_roles
False    317769
True      87736
Name: count, dtype: int64

Primary role:
credit_primary_role
None                          164585
Writing – Review & Editing     55686
Formal Analysis                46244
Investigation                  34578
Writin

In [ ]:
mapping_df_nature_actions_credit

,_mapping_row_id,dataset,source_folder,file_name,file_path,source_key,local_article_index,global_article_index,article_instance_key,title,...,credit_send_to_few_shot,credit_candidate_roles,credit_ambiguous_predicates,credit_normalized_task,credit_action_mappings_rule,credit_unresolved_actions_rule,credit_atomic_action_count,action_index,action_text,action_source
0,0,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,...,False,[],[],wrote the first draft,"[{""action_index"": 1, ""action_text"": ""wrote the...",[],1,1,wrote the first draft,original_task
1,0,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,...,False,[],[],undertook the data analysis,"[{""action_index"": 1, ""action_text"": ""undertook...",[],1,1,undertook the data analysis,original_task
2,0,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,...,False,[],[],edited the manuscript,"[{""action_index"": 1, ""action_text"": ""edited th...",[],1,1,edited the manuscript,original_task
3,1,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,...,True,"[""Investigation"", ""Data Curation""]",[],collected the datasets,"[{""action_index"": 1, ""action_text"": ""collected...","[""collected the datasets""]",1,1,collected the datasets,original_task
4,1,Nature,nature,humanities and social sciences communications...,/content/drive/MyDrive/data fro data_mining_pr...,1,1,1,nature|| humanities and social sciences commun...,AI chatbots contribute to global conservation ...,...,False,[],[],undertook the data analysis,"[{""action_index"": 1, ""action_text"": ""undertook...",[],1,1,undertook the data analysis,original_task
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357765,117143,Nature,nature,scientific data_updated.json,/content/drive/MyDrive/data fro data_mining_pr...,115,7,25965,nature||scientific data_updated.json||115||7,Multivariate time series dataset for space wea...,...,False,[],[],"contributed to the writing of the manuscript, ...","[{""action_index"": 1, ""action_text"": ""contribut...","[""contributed to created the X-ray flux series...",8,7,provided guidance flare integration and cleani...,coordinated_verbs
357766,117143,Nature,nature,scientific data_updated.json,/content/drive/MyDrive/data fro data_mining_pr...,115,7,25965,nature||scientific data_updated.json||115||7,Multivariate time series dataset for space wea...,...,True,[],[],"contributed to the writing of the manuscript, ...","[{""action_index"": 1, ""action_text"": ""contribut...","[""contributed to created the X-ray flux series...",8,8,created the X ray flux series,coordinated_verbs
357767,117144,Nature,nature,scientific data_updated.json,/content/drive/MyDrive/data fro data_mining_pr...,115,7,25965,nature||scientific data_updated.json||115||7,Multivariate time series dataset for space wea...,...,False,[],[],participated in data acquisition and creation ...,"[{""action_index"": 1, ""action_text"": ""participa...",[],1,1,participated in data acquisition and creation ...,original_task
357768,117145,Nature,nature,scientific data_updated.json,/content/drive/MyDrive/data fro data_mining_pr...,115,7,25965,nature||scientific data_updated.json||115||7,Multivariate time series dataset for space wea...,...,True,"[""Data Curation""]",[],participated in cleaning and integration of noaa,"[{""action_index"": 1, ""action_text""

In [ ]:
mapping_df_nature_credit.to_csv('mapping_df_nature_credit.csv')

# Few-shot Classification with Embeddings

## 23. Few-shot Example File Format

The example file must contain at least two columns:

| category | example |
|---|---|
| Formal Analysis | analyzed the experimental data |
| Investigation | performed laboratory experiments |
| Writing – Review & Editing | revised the manuscript |

Each row is one manually labeled example. Use verified real examples whenever possible rather than automatically generated examples.

CSV and Excel files are supported. Each category should contain several diverse examples.


In [ ]:
# Build category examples from all trusted rule-assigned atomic actions.

TRUSTED_EXAMPLE_METHODS = {
    "explicit_role",
    "explicit_multi_role",
    "text_rule",
    "text_multi_role",
    "dependency_rule",
    "dependency_multi_role",
    "nominal_rule",
    "nominal_multi_role",
    "atomic_text_rule",
    "atomic_shared_object_rule",
}

def parse_json_roles(value):
    """Return a clean list of valid CRediT roles from JSON or a Python list."""
    if isinstance(value, list):
        roles = value
    else:
        try:
            roles = json.loads(value)
        except (TypeError, json.JSONDecodeError):
            roles = []

    return [
        role
        for role in roles
        if role in CREDIT_ROLES
    ]


def build_examples_from_action_dataframe(actions_df):
    """
    Build one grouped example bank from trusted atomic-action assignments.

    Each multi-role action is copied to every assigned category.
    Duplicate text-category pairs are represented by occurrence_count.
    """
    required_columns = {
        "action_text",
        "credit_roles_rule",
        "credit_assignment_method",
        "credit_send_to_few_shot",
        "credit_needs_manual_review",
        "credit_special_type",
    }
    missing = required_columns - set(actions_df.columns)
    if missing:
        raise KeyError(
            "The atomic-action DataFrame is missing columns: "
            f"{sorted(missing)}"
        )

    source = actions_df.copy()
    source["roles_list"] = source["credit_roles_rule"].apply(
        parse_json_roles
    )

    trusted_mask = (
        source["roles_list"].map(bool)
        & source["credit_send_to_few_shot"].eq(False)
        & source["credit_needs_manual_review"].eq(False)
        & source["credit_assignment_method"].isin(
            TRUSTED_EXAMPLE_METHODS
        )
        & source["credit_special_type"].isna()
    )

    trusted = source.loc[
        trusted_mask,
        [
            "action_text",
            "roles_list",
            "credit_assignment_method",
        ],
    ].copy()

    trusted["example"] = (
        trusted["action_text"]
        .astype(str)
        .str.strip()
    )
    trusted = trusted[trusted["example"].ne("")]

    # One multi-role action becomes one example in every assigned role.
    exploded = trusted.explode("roles_list").rename(
        columns={"roles_list": "category"}
    )
    exploded = exploded[
        exploded["category"].isin(CREDIT_ROLES)
    ].copy()

    exploded["normalized_example"] = exploded["example"].map(
        normalize_task_text
    )
    exploded = exploded[
        exploded["normalized_example"].ne("")
    ]

    # Group repeated author occurrences without losing their frequency.
    grouped = (
        exploded.groupby(
            ["category", "normalized_example"],
            as_index=False,
        )
        .agg(
            example=("example", "first"),
            occurrence_count=("example", "size"),
            assignment_methods=(
                "credit_assignment_method",
                lambda values: json.dumps(
                    sorted(set(values)),
                    ensure_ascii=False,
                ),
            ),
        )
    )

    grouped["category_order"] = grouped["category"].map(
        {role: index for index, role in enumerate(CREDIT_ROLES)}
    )
    grouped = (
        grouped.sort_values(
            [
                "category_order",
                "occurrence_count",
                "normalized_example",
            ],
            ascending=[True, False, True],
        )
        .drop(columns="category_order")
        .reset_index(drop=True)
    )

    missing_roles = [
        role
        for role in CREDIT_ROLES
        if role not in set(grouped["category"])
    ]
    if missing_roles:
        raise ValueError(
            "No trusted examples were found for: "
            f"{missing_roles}"
        )

    return grouped


credit_examples_by_category_df = (
    build_examples_from_action_dataframe(
        mapping_df_nature_actions_credit
    )
)

# Convenient grouped dictionary for inspection.
category_examples = {
    role: credit_examples_by_category_df.loc[
        credit_examples_by_category_df["category"].eq(role),
        "example",
    ].tolist()
    for role in CREDIT_ROLES
}

example_bank_summary = (
    credit_examples_by_category_df.groupby(
        "category",
        as_index=False,
    )
    .agg(
        unique_examples=("example", "size"),
        total_occurrences=("occurrence_count", "sum"),
    )
    .set_index("category")
    .reindex(CREDIT_ROLES)
    .reset_index()
)

display(example_bank_summary)

# Show the most frequent examples in each category.
display(
    credit_examples_by_category_df.groupby(
        "category",
        group_keys=False,
    ).head(10)
)


,category,unique_examples,total_occurrences
0,Conceptualization,6845,35686
1,Methodology,8809,44104
2,Investigation,12223,58732
3,Formal Analysis,19152,101713
4,Data Curation,3728,11462
5,Writing – Original Draft,8589,64672
6,Writing – Review & Editing,7520,96065
7,Supervision,5572,21268
8,Validation,2838,7617
9,Project Administration,1355,4459


,category,normalized_example,example,occurrence_count,assignment_methods
0,Conceptualization,conceptualization,Conceptualization,3288,"[""explicit_role""]"
1,Conceptualization,conceived the study,conceived the study,3247,"[""text_rule""]"
2,Conceptualization,conceived the project,conceived the project,2134,"[""text_rule""]"
3,Conceptualization,conceived the idea,Conceived the idea,1944,"[""text_rule""]"
4,Conceptualization,conceived the experiments,conceived the experiments,1421,"[""text_rule""]"
...,...,...,...,...,...
84038,Software,contributed to software development,contributed to software development,39,"[""text_rule""]"
84039,Software,developed the software,developed the software,38,"[""text_rule""]"
84040,Software,implemented the model,implemented the model,36,"[""dependency_rule""]"
84041,Software,contributed to software,contributed to software,35,"[""text_rule""]"


In [ ]:
# Embed the automatically extracted category examples.

import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)
EMBEDDING_BATCH_SIZE = 128
USE_FREQUENCY_WEIGHTS = True

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

example_embeddings = embedding_model.encode(
    credit_examples_by_category_df["example"].tolist(),
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

credit_examples_by_category_df = (
    credit_examples_by_category_df.copy()
)
credit_examples_by_category_df["embedding_index"] = range(
    len(credit_examples_by_category_df)
)

role_centroids = {}
role_example_indices = {}

for role in CREDIT_ROLES:
    indices = credit_examples_by_category_df.index[
        credit_examples_by_category_df[
            "category"
        ].eq(role)
    ].to_numpy()

    if len(indices) == 0:
        continue

    role_vectors = example_embeddings[indices]

    if USE_FREQUENCY_WEIGHTS:
        weights = (
            credit_examples_by_category_df.loc[
                indices,
                "occurrence_count",
            ]
            .astype(float)
            .to_numpy()
        )
    else:
        weights = np.ones(len(indices), dtype=float)

    centroid = np.average(
        role_vectors,
        axis=0,
        weights=weights,
    )
    centroid_norm = np.linalg.norm(centroid)

    if centroid_norm == 0:
        continue

    role_centroids[role] = centroid / centroid_norm
    role_example_indices[role] = indices

CENTROID_ROLES = [
    role
    for role in CREDIT_ROLES
    if role in role_centroids
]
CENTROID_MATRIX = np.vstack(
    [
        role_centroids[role]
        for role in CENTROID_ROLES
    ]
)

print(
    "Categories with centroids:",
    len(CENTROID_ROLES),
)
print(
    "Unique text-category examples:",
    len(credit_examples_by_category_df),
)
print(
    "Total weighted occurrences:",
    int(
        credit_examples_by_category_df[
            "occurrence_count"
        ].sum()
    ),
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/677 [00:00<?, ?it/s]

Categories with centroids: 14
Unique text-category examples: 86650
Total weighted occurrences: 479499


## 25. Few-shot Classification Function

Building threshold for each category


In [ ]:
import json
import re
import numpy as np
import pandas as pd


def build_few_shot_role_thresholds(
    examples_df,
    embeddings,
    centroid_roles,
    centroid_matrix,
    use_frequency_weights=True,
    lower_quantile=0.05,
    threshold_slack=0.04,
):
    """
    Build category-specific thresholds using leave-one-out centroids.

    This avoids one arbitrary threshold for all CRediT categories.
    """
    examples = (
        examples_df
        .sort_values("embedding_index")
        .reset_index(drop=True)
        .copy()
    )

    if len(examples) != len(embeddings):
        raise ValueError(
            "Examples and example embeddings are not aligned."
        )

    if (
        use_frequency_weights
        and "occurrence_count" in examples.columns
    ):
        all_weights = (
            examples["occurrence_count"]
            .astype(float)
            .to_numpy()
        )
    else:
        all_weights = np.ones(
            len(examples),
            dtype=float,
        )

    role_to_column = {
        role: index
        for index, role in enumerate(centroid_roles)
    }

    fixed_scores = embeddings @ centroid_matrix.T
    thresholds = {}

    for role in centroid_roles:
        role_column = role_to_column[role]

        indices = examples.index[
            examples["category"].eq(role)
        ].to_numpy()

        if len(indices) == 0:
            continue

        vectors = embeddings[indices]
        weights = all_weights[indices]

        weighted_sum = (
            vectors * weights[:, None]
        ).sum(axis=0)

        total_weight = weights.sum()

        # Leave-one-out centroid for every training example.
        if len(indices) > 1:
            denominators = total_weight - weights

            loo_centroids = (
                weighted_sum[None, :]
                - weights[:, None] * vectors
            ) / denominators[:, None]

            norms = np.linalg.norm(
                loo_centroids,
                axis=1,
                keepdims=True,
            )

            loo_centroids = np.divide(
                loo_centroids,
                norms,
                out=np.zeros_like(loo_centroids),
                where=norms != 0,
            )

            own_scores = np.sum(
                vectors * loo_centroids,
                axis=1,
            )
        else:
            own_scores = fixed_scores[
                indices,
                role_column,
            ]

        other_scores = np.delete(
            fixed_scores[indices],
            role_column,
            axis=1,
        )

        best_other_scores = other_scores.max(
            axis=1
        )

        margins = own_scores - best_other_scores

        correctly_separated = margins > 0

        if correctly_separated.any():
            similarity_reference = own_scores[
                correctly_separated
            ]
            margin_reference = margins[
                correctly_separated
            ]
        else:
            similarity_reference = own_scores
            margin_reference = margins

        minimum_similarity = float(
            np.quantile(
                similarity_reference,
                lower_quantile,
            )
            - threshold_slack
        )

        minimum_margin = float(
            np.quantile(
                margin_reference,
                lower_quantile,
            )
            - 0.01
        )

        strong_similarity = float(
            np.quantile(
                similarity_reference,
                0.75,
            )
        )

        thresholds[role] = {
            "min_similarity": max(
                minimum_similarity,
                0.10,
            ),
            "min_margin": max(
                minimum_margin,
                0.005,
            ),
            "strong_similarity": (
                strong_similarity
            ),
            "example_count": int(len(indices)),
        }

    return thresholds


FEW_SHOT_ROLE_THRESHOLDS = (
    build_few_shot_role_thresholds(
        credit_examples_by_category_df,
        example_embeddings,
        CENTROID_ROLES,
        CENTROID_MATRIX,
        use_frequency_weights=USE_FREQUENCY_WEIGHTS,
    )
)


display(
    pd.DataFrame(
        FEW_SHOT_ROLE_THRESHOLDS
    ).T
)

,min_similarity,min_margin,strong_similarity,example_count
Conceptualization,0.365513,0.005000,0.718522,6845.0
Methodology,0.225902,0.005000,0.733099,8809.0
Investigation,0.164278,0.005000,0.619890,12223.0
Formal Analysis,0.196360,0.005000,0.672328,19152.0
Data Curation,0.336206,0.005000,0.721841,3728.0
Writing – Original Draft,0.490259,0.005000,0.779146,8589.0
Writing – Review & Editing,0.447708,0.005000,0.758043,7520.0
Supervision,0.315357,0.005000,0.708077,5572.0
Validation,0.335543,0.005000,0.720652,2838.0
Project Administration,0.486514,0.007813,0.776892,1355.0


## 27. Run Few-shot Classification Only on Unassigned  Actions

The embedding model is applied only to atomic actions where the deterministic rules found no role.

Classifying the atomic action rather than the entire original task prevents a recognized manuscript action from hiding an unresolved data-generation or analysis action.


In [ ]:
def split_compound_few_shot_task(
    text,
    nlp_model,
):
    """
    Split only when dependency parsing finds coordinated verbs.

    A comma or 'and' alone does not cause a split.
    """
    text = " ".join(
        str(text).strip().split()
    )

    if not re.search(
        r",|;|\b(?:and|or|as well as)\b",
        text,
        flags=re.I,
    ):
        return [text]

    doc = nlp_model(text)

    root_verbs = [
        token
        for token in doc
        if (
            token.dep_ == "ROOT"
            and token.pos_ == "VERB"
        )
    ]

    if not root_verbs:
        return [text]

    root = root_verbs[0]

    def belongs_to_root(token):
        current = token

        while current.dep_ == "conj":
            current = current.head

        return current.i == root.i

    verbs = [
        token
        for token in doc
        if (
            token.pos_ == "VERB"
            and (
                token.i == root.i
                or (
                    token.dep_ == "conj"
                    and belongs_to_root(token)
                )
            )
        )
    ]

    verbs = sorted(
        verbs,
        key=lambda token: token.i,
    )

    if len(verbs) < 2:
        return [text]

    parts = []

    for index, verb in enumerate(verbs):
        start = 0 if index == 0 else verb.i

        end = (
            verbs[index + 1].i
            if index + 1 < len(verbs)
            else len(doc)
        )

        part = doc[start:end].text.strip()

        part = re.sub(
            r"^[,;:\s]*(?:and|or|also|then|as well as)\s+",
            "",
            part,
            flags=re.I,
        )

        part = re.sub(
            r"\s+(?:and|or|also|then|as well as)[,;:\s]*$",
            "",
            part,
            flags=re.I,
        )

        part = part.strip(" ,;:.")

        if part:
            parts.append(part)

    # Recover a shared object:
    # "collected and analyzed the data"
    # -> "collected the data", "analyzed the data"
    last_verb = verbs[-1]

    shared_tail = doc[
        last_verb.i + 1:
    ].text.strip(" ,;:.")

    if shared_tail:
        for index in range(len(parts) - 1):
            part_doc = nlp_model(parts[index])

            has_object = any(
                token.dep_ in {
                    "obj",
                    "dobj",
                    "pobj",
                    "attr",
                    "dative",
                    "oprd",
                }
                or token.pos_ in {
                    "NOUN",
                    "PROPN",
                    "PRON",
                }
                for token in part_doc
            )

            if not has_object:
                parts[index] = (
                    f"{parts[index]} {shared_tail}"
                )

    parts = list(dict.fromkeys(parts))

    return parts if len(parts) >= 2 else [text]

In [ ]:
def classify_tasks_with_few_shot(
    tasks,
    model,
    role_thresholds=FEW_SHOT_ROLE_THRESHOLDS,
    nlp_model=None,
    allow_two_roles=True,
    batch_size=128,
):
    """
    Classify unresolved tasks using calibrated Few-shot embeddings.

    Atomic task:
        At most one role.

    Compound task:
        Each detected action receives Top-1.
        The original task may receive two different roles.
    """
    if nlp_model is None:
        nlp_model = nlp

    tasks = list(dict.fromkeys(
        str(task).strip()
        for task in tasks
        if str(task).strip()
    ))

    if not tasks:
        return pd.DataFrame()

    task_parts = {
        task: split_compound_few_shot_task(
            task,
            nlp_model,
        )
        for task in tasks
    }

    unique_parts = list(dict.fromkeys(
        part
        for parts in task_parts.values()
        for part in parts
    ))

    part_embeddings = model.encode(
        unique_parts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    embedding_lookup = {
        part: part_embeddings[index]
        for index, part in enumerate(unique_parts)
    }

    def classify_one_part(part):
        embedding = embedding_lookup[part]
        scores = embedding @ CENTROID_MATRIX.T

        order = np.argsort(-scores)

        top_index = int(order[0])
        second_index = int(order[1])

        top_role = CENTROID_ROLES[top_index]
        second_role = CENTROID_ROLES[
            second_index
        ]

        top_score = float(scores[top_index])
        second_score = float(
            scores[second_index]
        )

        score_gap = top_score - second_score

        thresholds = role_thresholds[
            top_role
        ]

        accepted = (
            top_score
            >= thresholds["min_similarity"]
            and (
                score_gap
                >= thresholds["min_margin"]
                or top_score
                >= thresholds[
                    "strong_similarity"
                ]
            )
        )

        example_indices = (
            role_example_indices[top_role]
        )

        example_scores = (
            example_embeddings[example_indices]
            @ embedding
        )

        nearest_local_index = int(
            np.argmax(example_scores)
        )

        nearest_example_index = int(
            example_indices[
                nearest_local_index
            ]
        )

        return {
            "part": part,
            "accepted": accepted,
            "top_role": top_role,
            "top_score": top_score,
            "second_role": second_role,
            "second_score": second_score,
            "margin": score_gap,
            "nearest_example": (
                credit_examples_by_category_df.loc[
                    nearest_example_index,
                    "example",
                ]
            ),
            "nearest_similarity": float(
                example_scores[
                    nearest_local_index
                ]
            ),
        }

    records = []

    for task in tasks:
        parts = task_parts[task]

        part_results = [
            classify_one_part(part)
            for part in parts
        ]

        all_parts_accepted = all(
            result["accepted"]
            for result in part_results
        )

        accepted_roles = [
            result["top_role"]
            for result in part_results
            if result["accepted"]
        ]

        selected_roles = [
            role
            for role in CREDIT_ROLES
            if role in accepted_roles
        ]

        is_compound = len(parts) > 1

        # A single atomic action can never receive two roles.
        if not is_compound:
            selected_roles = (
                selected_roles[:1]
            )

        # The fallback compound classifier may return at most two roles.
        if (
            is_compound
            and (
                not allow_two_roles
                or len(selected_roles) > 2
            )
        ):
            all_parts_accepted = False
            selected_roles = []

        accepted = (
            all_parts_accepted
            and bool(selected_roles)
        )

        if not accepted:
            final_roles = []
        else:
            final_roles = selected_roles

        rejection_reasons = []

        if not all_parts_accepted:
            rejection_reasons.append(
                "one_or_more_parts_uncertain"
            )

        if (
            is_compound
            and len(selected_roles) > 2
        ):
            rejection_reasons.append(
                "more_than_two_distinct_roles"
            )

        weakest_result = min(
            part_results,
            key=lambda result: result[
                "top_score"
            ],
        )

        minimum_margin_result = min(
            part_results,
            key=lambda result: result[
                "margin"
            ],
        )

        records.append({
            "task_single": task,

            "few_shot_roles": json.dumps(
                final_roles,
                ensure_ascii=False,
            ),

            "few_shot_primary_role": (
                final_roles[0]
                if len(final_roles) == 1
                else None
            ),

            "few_shot_accepted": accepted,

            "few_shot_is_compound": (
                is_compound
            ),

            "few_shot_parts": json.dumps(
                parts,
                ensure_ascii=False,
            ),

            "few_shot_part_roles": json.dumps(
                [
                    (
                        [result["top_role"]]
                        if result["accepted"]
                        else []
                    )
                    for result in part_results
                ],
                ensure_ascii=False,
            ),

            "few_shot_top_role": (
                final_roles[0]
                if final_roles
                else weakest_result[
                    "top_role"
                ]
            ),

            "few_shot_top_similarity": float(
                weakest_result["top_score"]
            ),

            "few_shot_top_distance": (
                1.0
                - float(
                    weakest_result[
                        "top_score"
                    ]
                )
            ),

            "few_shot_second_role": (
                final_roles[1]
                if len(final_roles) == 2
                else weakest_result[
                    "second_role"
                ]
            ),

            "few_shot_second_similarity": float(
                weakest_result[
                    "second_score"
                ]
            ),

            "few_shot_margin": float(
                minimum_margin_result[
                    "margin"
                ]
            ),

            "few_shot_nearest_example": (
                weakest_result[
                    "nearest_example"
                ]
            ),

            "few_shot_nearest_example_similarity": float(
                weakest_result[
                    "nearest_similarity"
                ]
            ),

            "few_shot_rejection_reason": (
                "|".join(rejection_reasons)
                if rejection_reasons
                else None
            ),
        })

    return pd.DataFrame(records)

In [ ]:
unresolved_actions = (
    mapping_df_nature_actions_credit.loc[
        mapping_df_nature_actions_credit[
            "credit_send_to_few_shot"
        ].eq(True),
        "action_text",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .tolist()
)

print("Unique unresolved actions:", len(unresolved_actions))
few_shot_predictions_df = (
    classify_tasks_with_few_shot(
        unresolved_actions,
        embedding_model,
        role_thresholds=(
            FEW_SHOT_ROLE_THRESHOLDS
        ),
        nlp_model=nlp,
        allow_two_roles=True,
        batch_size=EMBEDDING_BATCH_SIZE,
    )
    .rename(
        columns={
            "task_single": "action_text"
        }
    )
)

Unique unresolved actions: 28769


Batches:   0%|          | 0/226 [00:00<?, ?it/s]

## 28. Merge Few-shot Predictions Back into Original Tasks

First, rule and embedding results are combined at the atomic-action level.

The atomic actions are then aggregated back to each original task. The final task-level output contains:

- the unique CRediT roles of the task;
- every detected action;
- the role of every action;
- whether any action remains unresolved.


In [ ]:

actions_credit_final_df = mapping_df_nature_actions_credit.merge(
    few_shot_predictions_df,
    on="action_text",
    how="left",
)


def finalize_atomic_action(row):
    """Choose rule roles first and use embeddings only for unresolved actions."""
    rule_roles = json.loads(row["credit_roles_rule"])
    was_sent = bool(row["credit_send_to_few_shot"])
    accepted = (
        bool(row["few_shot_accepted"])
        if pd.notna(row.get("few_shot_accepted"))
        else False
    )

    if was_sent and accepted:
        final_roles = json.loads(row["few_shot_roles"])
        method = "few_shot_embedding"
        confidence = float(row["few_shot_top_similarity"])
        still_unresolved = False
    else:
        final_roles = rule_roles
        method = row["credit_assignment_method"]
        confidence = row.get("credit_rule_confidence", 0.0)
        still_unresolved = was_sent

    return pd.Series({
        "credit_roles_final": json.dumps(
            final_roles,
            ensure_ascii=False,
        ),
        "credit_primary_role_final": (
            final_roles[0]
            if len(final_roles) == 1
            else None
        ),
        "credit_assignment_method_final": method,
        "credit_confidence_final": confidence,
        "credit_contains_multiple_roles_final": (
            len(final_roles) > 1
        ),
        "credit_send_to_few_shot_final": still_unresolved,
    })


action_final_columns = actions_credit_final_df.apply(
    finalize_atomic_action,
    axis=1,
)
actions_credit_final_df = pd.concat(
    [actions_credit_final_df, action_final_columns],
    axis=1,
)


def aggregate_atomic_actions(group):
    """Aggregate atomic actions back to one original task."""
    ordered_group = group.sort_values("action_index")
    final_roles = ordered_roles(
        role
        for value in ordered_group["credit_roles_final"]
        for role in json.loads(value)
    )

    action_mapping = []
    for _, row in ordered_group.iterrows():
        action_mapping.append({
            "action_index": int(row["action_index"]),
            "action_text": row["action_text"],
            "source": row["action_source"],
            "roles": json.loads(row["credit_roles_final"]),
            "assignment_method": row[
                "credit_assignment_method_final"
            ],
            "confidence": row["credit_confidence_final"],
            "still_unresolved": bool(
                row["credit_send_to_few_shot_final"]
            ),
            "nearest_example": row.get(
                "few_shot_nearest_example"
            ),
            "similarity": row.get(
                "few_shot_top_similarity"
            ),
        })

    return pd.Series({
        "credit_roles_final": json.dumps(
            final_roles,
            ensure_ascii=False,
        ),
        "credit_primary_role_final": (
            final_roles[0]
            if len(final_roles) == 1
            else None
        ),
        "credit_action_mapping_final": json.dumps(
            action_mapping,
            ensure_ascii=False,
        ),
        "credit_atomic_action_count_final": len(action_mapping),
        "credit_send_to_few_shot_final": bool(
            ordered_group[
                "credit_send_to_few_shot_final"
            ].any()
        ),
        "credit_assignment_method_final": (
            "action_aware_with_few_shot"
            if any(
                ordered_group[
                    "credit_assignment_method_final"
                ].eq("few_shot_embedding")
            )
            else "action_aware_rule"
        ),
    })


task_key_columns = [
    "_mapping_row_id",
    "task_position",
    "task_single",
]

task_final_summary = (
    actions_credit_final_df
    .groupby(task_key_columns, as_index=False, sort=False)
    .apply(
        aggregate_atomic_actions,
        include_groups=False,
    )
    .reset_index()
)

mapping_df_nature_tasks_final = (
    mapping_df_nature_tasks_credit
    .drop(
        columns=[
            "credit_roles_final",
            "credit_primary_role_final",
            "credit_action_mapping_final",
            "credit_atomic_action_count_final",
            "credit_send_to_few_shot_final",
            "credit_assignment_method_final",
        ],
        errors="ignore",
    )
    .merge(
        task_final_summary,
        on=task_key_columns,
        how="left",
    )
)

display(
    mapping_df_nature_tasks_final[
        [
            "task_single",
            "credit_roles_final",
            "credit_action_mapping_final",
            "credit_send_to_few_shot_final",
        ]
    ].head(30)
)


,task_single,credit_roles_final,credit_action_mapping_final,credit_send_to_few_shot_final
0,wrote the first draft,"[""Writing – Original Draft""]","[{""action_index"": 1, ""action_text"": ""wrote the...",False
1,undertook the data analysis,"[""Formal Analysis""]","[{""action_index"": 1, ""action_text"": ""undertook...",False
2,edited the manuscript,"[""Writing – Review & Editing""]","[{""action_index"": 1, ""action_text"": ""edited th...",False
3,collected the datasets,"[""Data Curation""]","[{""action_index"": 1, ""action_text"": ""collected...",False
4,undertook the data analysis,"[""Formal Analysis""]","[{""action_index"": 1, ""action_text"": ""undertook...",False
5,assisted in developing the conceptual and anal...,"[""Writing – Review & Editing""]","[{""action_index"": 1, ""action_text"": ""commented...",False
6,assisted in developing the conceptual and anal...,"[""Writing – Review & Editing""]","[{""action_index"": 1, ""action_text"": ""commented...",False
7,edited the manuscript,"[""Writing – Review & Editing""]","[{""action_index"": 1, ""action_text"": ""edited th...",False
8,wrote the original manuscript of this study,"[""Writing – Original Draft""]","[{""action_index"": 1, ""action_text"": ""wrote the...",False
9,responsible for the revision of the article,"[""Writing – Review & Editing""]","[{""action_index"": 1, ""action_text"": ""responsib...",False


## 29. Rebuild the Author-level Output

This cell aggregates the final task and action results back to each author while preserving the detailed action mapping for every original task.


In [ ]:

def aggregate_json_role_values(values):
    """Return unique CRediT roles in the fixed category order."""
    return ordered_roles(
        role
        for value in values
        for role in json.loads(value)
    )


author_records = []

for row_id, group in mapping_df_nature_tasks_final.groupby(
    "_mapping_row_id",
    sort=False,
):
    ordered_group = group.sort_values("task_position")
    final_roles = aggregate_json_role_values(
        ordered_group["credit_roles_final"]
    )

    author_records.append({
        "_mapping_row_id": row_id,
        "credit_all_roles_final": json.dumps(
            final_roles,
            ensure_ascii=False,
        ),
        "credit_action_mappings_per_task_final": json.dumps(
            [
                json.loads(value)
                for value in ordered_group[
                    "credit_action_mapping_final"
                ]
            ],
            ensure_ascii=False,
        ),
        "credit_any_few_shot_assigned": bool(
            ordered_group[
                "credit_assignment_method_final"
            ].eq("action_aware_with_few_shot").any()
        ),
        "credit_remaining_unresolved_count": int(
            ordered_group[
                "credit_send_to_few_shot_final"
            ].sum()
        ),
    })

author_final_summary = pd.DataFrame(author_records)

mapping_credit_final_df = mapping_df_nature_credit.merge(
    author_final_summary,
    on="_mapping_row_id",
    how="left",
)

display(
    mapping_credit_final_df[
        [
            "full_name",
            "tasks",
            "credit_all_roles_final",
            "credit_any_few_shot_assigned",
            "credit_remaining_unresolved_count",
        ]
    ].head(20)
)


,full_name,tasks,credit_all_roles_final,credit_any_few_shot_assigned,credit_remaining_unresolved_count
0,Danilo Urzedo,wrote the first draft | undertook the data ana...,"[""Formal Analysis"", ""Writing – Original Draft""...",False,0.0
1,Zarrin Tasnim Sworna,collected the datasets | undertook the data an...,"[""Formal Analysis"", ""Data Curation""]",True,0.0
2,Andrew J. Hoskins,assisted in developing the conceptual and anal...,"[""Writing – Review & Editing""]",False,0.0
3,Cathy J. Robinson,assisted in developing the conceptual and anal...,"[""Writing – Review & Editing""]",False,0.0
4,Hyeon Jo,wrote the original manuscript of this study,"[""Writing – Original Draft""]",False,0.0
5,Youngsok Bang,responsible for the revision of the article,"[""Writing – Review & Editing""]",False,0.0
6,Rameez Raja,"actively participated in the writing process, ...","[""Conceptualization"", ""Methodology"", ""Investig...",False,1.0
7,Jianfu Ma,"actively participated in the writing process, ...","[""Investigation"", ""Formal Analysis"", ""Writing ...",True,1.0
8,Rui Tao,"actively participated in the writing process, ...","[""Investigation"", ""Writing – Original Draft"", ...",True,1.0
9,Shakir Ullah,"actively participated in the writing process, ...","[""Methodology"", ""Writing – Original Draft"", ""W...",False,1.0


In [ ]:
total_tasks = mapping_credit_final_df[
    "credit_task_count"
].sum()

unresolved_tasks = mapping_credit_final_df[
    "credit_remaining_unresolved_count"
].fillna(0).sum()

print("Total author-task instances:", int(total_tasks))
print("Unresolved author-task instances:", int(unresolved_tasks))
print(
    "Unresolved percentage:",
    round(100 * unresolved_tasks / total_tasks, 2),
    "%"
)

Total author-task instances: 405505
Unresolved author-task instances: 26636
Unresolved percentage: 6.57 %


## 30. Final Summary and Save Output Files

Four outputs are saved for the first DataFrame:

1. Author-level final results.
2. Original author–task-level final results.
3. Atomic-action-level final results.
4. Few-shot predictions and similarity diagnostics.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files


# Save temporarily inside the Colab runtime, not Google Drive.
OUTPUT_DIR = Path("/content/credit_outputs2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


AUTHOR_OUTPUT_PATH = (
    OUTPUT_DIR / "mapping_credit_final.csv"
)
TASK_OUTPUT_PATH = (
    OUTPUT_DIR / "tasks_credit_final.csv"
)
ACTION_OUTPUT_PATH = (
    OUTPUT_DIR / "actions_credit_final.csv"
)
FEW_SHOT_OUTPUT_PATH = (
    OUTPUT_DIR / "few_shot_predictions.csv"
)


# Save all DataFrames.
mapping_credit_final_df.to_csv(
    AUTHOR_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

mapping_df_nature_tasks_final.to_csv(
    TASK_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

actions_credit_final_df.to_csv(
    ACTION_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

few_shot_predictions_df.to_csv(
    FEW_SHOT_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("Remaining unresolved atomic actions:")
print(
    int(
        actions_credit_final_df[
            "credit_send_to_few_shot_final"
        ].sum()
    )
)


print("\nSaved files:")
print(AUTHOR_OUTPUT_PATH)
print(TASK_OUTPUT_PATH)
print(ACTION_OUTPUT_PATH)
print(FEW_SHOT_OUTPUT_PATH)


# Create one ZIP containing all output files.
ZIP_BASE_PATH = "/content/credit_outputs2"

ZIP_PATH = shutil.make_archive(
    base_name=ZIP_BASE_PATH,
    format="zip",
    root_dir=OUTPUT_DIR,
)


print("\nZIP created:")
print(ZIP_PATH)


# Download the ZIP directly to the computer.
files.download(ZIP_PATH)

Remaining unresolved atomic actions:
28382

Saved files:
/content/credit_outputs2/mapping_credit_final.csv
/content/credit_outputs2/tasks_credit_final.csv
/content/credit_outputs2/actions_credit_final.csv
/content/credit_outputs2/few_shot_predictions.csv

ZIP created:
/content/credit_outputs2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

saving the few shot model

In [ ]:
from pathlib import Path
from google.colab import files

import json
import shutil
import numpy as np


OUTPUT_DIR = Path("/content/credit_centroid_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 1. Save centroids and example embeddings.
np.savez_compressed(
    OUTPUT_DIR / "centroids_and_embeddings.npz",
    centroid_matrix=CENTROID_MATRIX.astype(np.float32),
    example_embeddings=example_embeddings.astype(np.float32),
)


# 2. Save every example and its category assignment.
credit_examples_by_category_df.to_csv(
    OUTPUT_DIR / "credit_examples_by_category.csv",
    index=False,
    encoding="utf-8-sig",
)


# 3. Save the category orders.
with open(
    OUTPUT_DIR / "roles.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "credit_roles": list(CREDIT_ROLES),
            "centroid_roles": list(CENTROID_ROLES),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )


# 4. Save the example indices of every category.
role_indices_to_save = {
    role: [
        int(index)
        for index in indices
    ]
    for role, indices in role_example_indices.items()
}

with open(
    OUTPUT_DIR / "role_example_indices.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        role_indices_to_save,
        file,
        ensure_ascii=False,
        indent=2,
    )


# 5. Save the category-specific thresholds.
thresholds_to_save = {
    role: {
        key: (
            int(value)
            if isinstance(value, (np.integer,))
            else float(value)
            if isinstance(value, (np.floating,))
            else value
        )
        for key, value in role_values.items()
    }
    for role, role_values
    in FEW_SHOT_ROLE_THRESHOLDS.items()
}

with open(
    OUTPUT_DIR / "few_shot_role_thresholds.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        thresholds_to_save,
        file,
        ensure_ascii=False,
        indent=2,
    )


# 6. Save model configuration.
config = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "embedding_batch_size": int(EMBEDDING_BATCH_SIZE),
    "use_frequency_weights": bool(USE_FREQUENCY_WEIGHTS),
    "embedding_dimension": int(CENTROID_MATRIX.shape[1]),
    "number_of_centroids": int(CENTROID_MATRIX.shape[0]),
    "number_of_examples": int(example_embeddings.shape[0]),
    "normalized_embeddings": True,
    "normalized_centroids": True,
}

with open(
    OUTPUT_DIR / "config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        config,
        file,
        ensure_ascii=False,
        indent=2,
    )


# 7. Save the SentenceTransformer model itself.
embedding_model.save(
    str(OUTPUT_DIR / "sentence_transformer_model")
)


ZIP_PATH = shutil.make_archive(
    base_name="/content/credit_centroid_model",
    format="zip",
    root_dir=OUTPUT_DIR,
)


print("Saved package:")
print(ZIP_PATH)

print("\nFiles:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))


files.download(ZIP_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved package:
/content/credit_centroid_model.zip

Files:
centroids_and_embeddings.npz
config.json
credit_examples_by_category.csv
few_shot_role_thresholds.json
role_example_indices.json
roles.json
sentence_transformer_model/1_Pooling/config.json
sentence_transformer_model/README.md
sentence_transformer_model/config.json
sentence_transformer_model/config_sentence_transformers.json
sentence_transformer_model/model.safetensors
sentence_transformer_model/modules.json
sentence_transformer_model/sentence_bert_config.json
sentence_transformer_model/tokenizer.json
sentence_transformer_model/tokenizer_config.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Process a Plos (Contribution DF) DataFrame

## 31. Create a New Correction Column from Scratch

The second DataFrame stores contribution tasks as a list in the misspelled source column:

`contribtuion type`

Any existing correction column is explicitly removed and is not used for rules, examples, validation, comparison, or output construction.

The new column is named exactly:

`contribution type correction`

It contains one role list for every original task, in the same order as the source list.

Example:

```python
[
    "Conceived and designed the experiments",
    "wrote the paper",
]
```

becomes:

```python
[
    ["Conceptualization", "Methodology"],
    ["Writing – Original Draft"],
]
```

A second column, `contribution type action correction`, preserves every atomic action. This is necessary when one original task contains several actions or when two actions receive the same role.


In [ ]:
PLOS_ONE_INPUT_PATH = Path(
    "/content/df_with_openalex_fields.zip"
)

CENTROID_ZIP_PATH = Path(
    "/content/credit_centroid_model.zip"
)

CENTROID_EXTRACT_DIR = Path(
    "/content/credit_centroid_model_loaded"
)

OUTPUT_DIR = Path(
    "/content/plos_one_credit_rebuilt"
)


Load Few Shot Model:

In [ ]:
if CENTROID_EXTRACT_DIR.exists():
    shutil.rmtree(CENTROID_EXTRACT_DIR)

CENTROID_EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    CENTROID_ZIP_PATH,
    "r",
) as zip_file:
    zip_file.extractall(
        CENTROID_EXTRACT_DIR
    )

CENTROID_MODEL_DIR = next(
    CENTROID_EXTRACT_DIR.rglob(
        "centroids_and_embeddings.npz"
    )
).parent


saved_arrays = np.load(
    CENTROID_MODEL_DIR
    / "centroids_and_embeddings.npz",
    allow_pickle=False,
)

CENTROID_MATRIX = saved_arrays[
    "centroid_matrix"
].astype(np.float32)

example_embeddings = saved_arrays[
    "example_embeddings"
].astype(np.float32)


with open(
    CENTROID_MODEL_DIR / "roles.json",
    encoding="utf-8",
) as file:
    roles_data = json.load(file)

CREDIT_ROLES = roles_data["credit_roles"]
CENTROID_ROLES = roles_data["centroid_roles"]


credit_examples_by_category_df = (
    pd.read_csv(
        CENTROID_MODEL_DIR
        / "credit_examples_by_category.csv",
        encoding="utf-8-sig",
    )
    .sort_values("embedding_index")
    .reset_index(drop=True)
)


with open(
    CENTROID_MODEL_DIR
    / "role_example_indices.json",
    encoding="utf-8",
) as file:
    saved_indices = json.load(file)

role_example_indices = {
    role: np.asarray(indices, dtype=int)
    for role, indices in saved_indices.items()
}


with open(
    CENTROID_MODEL_DIR
    / "few_shot_role_thresholds.json",
    encoding="utf-8",
) as file:
    FEW_SHOT_ROLE_THRESHOLDS = json.load(file)


with open(
    CENTROID_MODEL_DIR / "config.json",
    encoding="utf-8",
) as file:
    model_config = json.load(file)

EMBEDDING_BATCH_SIZE = int(
    model_config["embedding_batch_size"]
)


embedding_model = SentenceTransformer(
    str(
        CENTROID_MODEL_DIR
        / "sentence_transformer_model"
    )
)

print("Centroid model loaded")
print("Categories:", len(CENTROID_ROLES))
print("Examples:", len(credit_examples_by_category_df))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Centroid model loaded
Categories: 14
Examples: 86650


## 32. Load the Second DataFrame

The example below reads a CSV directly from a ZIP archive containing one CSV file.

The existing `contribtuion type_correction` column may be present in the file, but the processing function removes it before creating the new correction column.


In [ ]:
def parse_task_list(value):
    """Parse the list stored in contribtuion type."""
    return [
        str(task).strip()
        for task in ast.literal_eval(value)
        if str(task).strip()
    ]


def parse_json_list(value):
    """Convert a saved JSON list to a Python list."""
    if isinstance(value, list):
        return value

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    try:
        result = json.loads(value)
        return result if isinstance(result, list) else []
    except Exception:
        return []


def safe_bool(value):
    """Convert a nullable value to Boolean."""
    if value is None:
        return False

    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass

    return bool(value)


def finalize_plos_one_action(row):
    """Use rules first and accepted Few-shot results second."""
    rule_roles = parse_json_list(
        row["credit_roles_rule"]
    )

    was_sent = safe_bool(
        row["credit_send_to_few_shot"]
    )

    accepted = safe_bool(
        row.get("few_shot_accepted")
    )

    if not was_sent:
        final_roles = rule_roles
        method = row["credit_assignment_method"]
        confidence = row["credit_rule_confidence"]
        unresolved = False

    elif accepted:
        final_roles = parse_json_list(
            row.get("few_shot_roles")
        )

        method = (
            "few_shot_compound_embedding"
            if safe_bool(
                row.get("few_shot_is_compound")
            )
            else "few_shot_embedding"
        )

        confidence = row.get(
            "few_shot_top_similarity",
            0.0,
        )

        unresolved = False

    else:
        final_roles = []
        method = "unresolved_after_few_shot"
        confidence = row.get(
            "few_shot_top_similarity",
            0.0,
        )
        unresolved = True

    if pd.isna(confidence):
        confidence = 0.0

    return pd.Series({
        "credit_roles_final": json.dumps(
            final_roles,
            ensure_ascii=False,
        ),
        "credit_assignment_method_final": method,
        "credit_confidence_final": float(confidence),
        "credit_unresolved_final": unresolved,
    })


In [ ]:
plos_one_df = pd.read_csv(
    PLOS_ONE_INPUT_PATH,
    compression="zip",
    encoding="utf-8-sig",
    low_memory=False,
)

plos_one_df = plos_one_df.drop(
    columns=[
        "contribution type correction",
        "contribution type action correction",
        "contribution type unresolved actions",
        "contribution type few shot actions",
        "contribution type few shot roles",
    ],
    errors="ignore",
)

plos_one_df["tasks_parsed"] = (
    plos_one_df["contribtuion type"]
    .apply(parse_task_list)
)

print("Rows:", f"{len(plos_one_df):,}")
print(
    "Task instances:",
    f"{plos_one_df['tasks_parsed'].map(len).sum():,}",
)

display(
    plos_one_df[
        [
            "title",
            "contribtuion type",
            "tasks_parsed",
        ]
    ].head()
)


Rows: 206,122
Task instances: 1,173,564


,title,contribtuion type,tasks_parsed
0,A Comparison Study of Single-Echo Susceptibili...,"['Conceived and designed the experiments', 'Pe...","[Conceived and designed the experiments, Perfo..."
1,Genetic Variation in the Mcp-1 Gene Promoter A...,"['Conceived and designed the experiments', 'Pe...","[Conceived and designed the experiments, Perfo..."
2,A Single cis Element Maintains Repression of t...,"['Conceived and designed the experiments', 'Pe...","[Conceived and designed the experiments, Perfo..."
3,Effects of Deworming on Malnourished Preschool...,"['Conceived and designed the experiments', 'Pe...","[Conceived and designed the experiments, Perfo..."
4,Involvement of Pancreatic Stellate Cells in Re...,"['Conceptualization', 'Data curation', 'Formal...","[Conceptualization, Data curation, Formal anal..."


In [ ]:
unique_tasks = list(dict.fromkeys(
    task
    for task_list in plos_one_df["tasks_parsed"]
    for task in task_list
))

print("Unique tasks:", f"{len(unique_tasks):,}")


rule_lookup = build_action_aware_lookup(
    unique_tasks,
    batch_size=256,
)


unique_action_records = []

for task in unique_tasks:
    task_result = rule_lookup[task]

    for action in task_result[
        "action_mappings_rule"
    ]:
        unique_action_records.append({
            "task_single": task,
            "action_index": action["action_index"],
            "action_text": action["action_text"],
            "action_source": action["source"],
            "credit_roles_rule": json.dumps(
                action["roles_rule"],
                ensure_ascii=False,
            ),
            "credit_assignment_method": action[
                "assignment_method_rule"
            ],
            "credit_rule_confidence": (
                0.98
                if action["roles_rule"]
                else 0.0
            ),
            "credit_send_to_few_shot": action[
                "send_to_few_shot"
            ],
            "credit_candidate_roles": json.dumps(
                action["candidate_roles"],
                ensure_ascii=False,
            ),
        })


unique_actions_df = pd.DataFrame(
    unique_action_records
)

print(
    "Unique task-action rows:",
    f"{len(unique_actions_df):,}",
)


Unique tasks: 30,815
Unique task-action rows: 40,109


## 33. Run Rules and Few-shot on the Second DataFrame

The same deterministic rules, action-aware layer, embeddings, and threshold are used.

Run Few-shot only on unresolved unique actions


In [ ]:
unresolved_action_texts = list(dict.fromkeys(
    unique_actions_df.loc[
        unique_actions_df[
            "credit_send_to_few_shot"
        ].eq(True),
        "action_text",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda values: values.ne("")]
    .tolist()
))

print(
    "Unique actions sent to Few-shot:",
    f"{len(unresolved_action_texts):,}",
)


few_shot_predictions_df = (
    classify_tasks_with_few_shot(
        unresolved_action_texts,
        embedding_model,
        role_thresholds=(
            FEW_SHOT_ROLE_THRESHOLDS
        ),
        nlp_model=nlp,
        allow_two_roles=True,
        batch_size=EMBEDDING_BATCH_SIZE,
    )
    .rename(
        columns={
            "task_single": "action_text"
        }
    )
)


Unique actions sent to Few-shot: 13,697


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

In [ ]:
unique_actions_final_df = (
    unique_actions_df.merge(
        few_shot_predictions_df,
        on="action_text",
        how="left",
        validate="many_to_one",
    )
)


final_columns = unique_actions_final_df.apply(
    finalize_plos_one_action,
    axis=1,
)

unique_actions_final_df = pd.concat(
    [
        unique_actions_final_df,
        final_columns,
    ],
    axis=1,
)


task_result_lookup = {}

for task, group in unique_actions_final_df.groupby(
    "task_single",
    sort=False,
):
    group = group.sort_values(
        "action_index"
    )

    task_roles = ordered_roles(
        role
        for roles_json in group[
            "credit_roles_final"
        ]
        for role in parse_json_list(
            roles_json
        )
    )

    action_details = []

    for _, action in group.iterrows():
        action_details.append({
            "action_index": int(
                action["action_index"]
            ),
            "action_text": action["action_text"],
            "roles": parse_json_list(
                action["credit_roles_final"]
            ),
            "assignment_method": action[
                "credit_assignment_method_final"
            ],
            "confidence": float(
                action["credit_confidence_final"]
            ),
            "few_shot_parts": parse_json_list(
                action.get("few_shot_parts")
            ),
            "few_shot_part_roles": parse_json_list(
                action.get(
                    "few_shot_part_roles"
                )
            ),
            "unresolved": safe_bool(
                action[
                    "credit_unresolved_final"
                ]
            ),
        })

    task_result_lookup[task] = {
        "roles": task_roles,
        "actions": action_details,
        "few_shot_actions": [
            action["action_text"]
            for action in action_details
            if action["assignment_method"] in {
                "few_shot_embedding",
                "few_shot_compound_embedding",
            }
        ],
        "few_shot_roles": [
            action["roles"]
            for action in action_details
            if action["assignment_method"] in {
                "few_shot_embedding",
                "few_shot_compound_embedding",
            }
        ],
        "unresolved_actions": [
            action["action_text"]
            for action in action_details
            if action["unresolved"]
        ],
    }


## 35. Save the Corrected Second DataFrame

Rebuild the correction columns from contribtuion type

Save the new DF


In [ ]:
def build_row_corrections(task_list):
    """Build all new columns for one original PLOS ONE row."""
    task_roles = []
    task_action_details = []
    few_shot_actions = []
    few_shot_roles = []
    unresolved_actions = []

    for task_position, task in enumerate(
        task_list,
        start=1,
    ):
        result = task_result_lookup[task]

        task_roles.append(
            result["roles"]
        )

        task_action_details.append({
            "task_position": task_position,
            "original_task": task,
            "task_roles": result["roles"],
            "actions": result["actions"],
        })

        few_shot_actions.extend(
            result["few_shot_actions"]
        )

        few_shot_roles.extend(
            result["few_shot_roles"]
        )

        unresolved_actions.extend(
            result["unresolved_actions"]
        )

    return pd.Series({
        "contribution type correction": task_roles,
        "contribution type action correction": (
            task_action_details
        ),
        "contribution type few shot actions": (
            few_shot_actions
        ),
        "contribution type few shot roles": (
            few_shot_roles
        ),
        "contribution type unresolved actions": (
            unresolved_actions
        ),
    })


new_correction_columns = (
    plos_one_df["tasks_parsed"]
    .apply(build_row_corrections)
)

plos_one_corrected_df = pd.concat(
    [
        plos_one_df.drop(
            columns=["tasks_parsed"]
        ),
        new_correction_columns,
    ],
    axis=1,
)


In [ ]:
total_task_instances = int(
    plos_one_df[
        "tasks_parsed"
    ].map(len).sum()
)

unresolved_task_instances = int(
    plos_one_corrected_df[
        "contribution type unresolved actions"
    ].map(len).sum()
)

few_shot_action_instances = int(
    plos_one_corrected_df[
        "contribution type few shot actions"
    ].map(len).sum()
)

print(
    "Total task instances:",
    f"{total_task_instances:,}",
)

print(
    "Few-shot action instances:",
    f"{few_shot_action_instances:,}",
)

print(
    "Unresolved action instances:",
    f"{unresolved_task_instances:,}",
)

display(
    plos_one_corrected_df[
        [
            "contribtuion type",
            "contribution type correction",
            "contribution type few shot actions",
            "contribution type few shot roles",
            "contribution type unresolved actions",
        ]
    ].head(10)
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

plos_one_corrected_df.to_csv(
    OUTPUT_DIR
    / "plos_one_contribution_credit_rebuilt.csv",
    index=False,
    encoding="utf-8-sig",
)

unique_actions_final_df.to_csv(
    OUTPUT_DIR
    / "plos_one_unique_actions_credit_final.csv",
    index=False,
    encoding="utf-8-sig",
)

few_shot_predictions_df.to_csv(
    OUTPUT_DIR
    / "plos_one_few_shot_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)


OUTPUT_ZIP_PATH = shutil.make_archive(
    "/content/plos_one_credit_rebuilt",
    "zip",
    OUTPUT_DIR,
)

print(OUTPUT_ZIP_PATH)

files.download(
    OUTPUT_ZIP_PATH
)


Total task instances: 1,173,564
Few-shot action instances: 18,174
Unresolved action instances: 8,531


,contribtuion type,contribution type correction,contribution type few shot actions,contribution type few shot roles,contribution type unresolved actions
0,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
1,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
2,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
3,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
4,"['Conceptualization', 'Data curation', 'Formal...","[[Conceptualization], [Data Curation], [Formal...",[],[],[]
5,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
6,"['conceived and designed the experiments', 'pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
7,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
8,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]
9,"['Conceived and designed the experiments', 'Pe...","[[Conceptualization, Methodology], [Investigat...",[],[],[]


/content/plos_one_credit_rebuilt.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>